In [1]:
from connectivity_matrix_functions import find_poly, create_shp_index, calc_connectivity_lh, write_to_file, plot_conn_mat, plot_conn_mat_norm, plot_conn_mat_norm_noAN, plot_conn_mat_normcol_noAN, plot_conn_mat_diff
import pandas as pd
import geopandas as gpd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FormatStrFormatter
import cartopy
import cartopy.crs as ccrs
import time
import datetime
from netCDF4 import Dataset

In [ ]:
# Figure 1

In [ ]:
# name of input/output files and location they are saved:
particle_input_file_SL = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_SL_uniform_Cop_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_India = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_India_uniform_Cop_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_Bang = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Bang_uniform_Cop_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_Myan = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Myan_uniform_Cop_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_Thai = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Thai_uniform_Cop_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_Indo = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Indonesia_uniform_Cop_daily_Jun2018-Sept2019_monsoon.nc'
velocity_input_file = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/ocean_velocities_Cop_Jun2018-Sept2019_daily.nc'

# import data
openfile_p9 = Dataset(particle_input_file_SL)
lon_p_SL = openfile_p9.variables['lon']
lat_p_SL = openfile_p9.variables['lat']
time_p_SL = openfile_p9.variables['time']
openfile_p10 = Dataset(particle_input_file_India)
lon_p_India = openfile_p10.variables['lon']
lat_p_India = openfile_p10.variables['lat']
openfile_p11 = Dataset(particle_input_file_Bang)
lon_p_Bang = openfile_p11.variables['lon']
lat_p_Bang = openfile_p11.variables['lat']
openfile_p12 = Dataset(particle_input_file_Myan)
lon_p_Myan = openfile_p12.variables['lon']
lat_p_Myan = openfile_p12.variables['lat']
openfile_p13 = Dataset(particle_input_file_Thai)
lon_p_Thai = openfile_p13.variables['lon']
lat_p_Thai = openfile_p13.variables['lat']
openfile_p14 = Dataset(particle_input_file_Indo)
lon_p_Indo = openfile_p14.variables['lon']
lat_p_Indo = openfile_p14.variables['lat']

In [ ]:
# Copernicus
velocity_input_file54 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/ocean_velocities_Cop_Jun2018-Sept2019_daily.nc'

openfile_u54 = Dataset(velocity_input_file54)
lon_vec_u54 = openfile_u54.variables['longitude']
lat_vec_u54 = openfile_u54.variables['latitude']
lon_grid_u54 = lon_vec_u54
lat_grid_u54 = lat_vec_u54
time_u54 = openfile_u54.variables['time']
u54 =  openfile_u54.variables['uo']
v54 =  openfile_u54.variables['vo']

u_subset54= u54[53,0,:,:]
v_subset54= v54[53,0,:,:]
speed_subset54 = np.sqrt(np.abs(u_subset54)**2 + np.abs(v_subset54)**2)

interval_type54 = 'hours'
interval_num54 = int(time_u54[53])

origin_time_u54 = datetime.datetime.strptime('1950/01/01 00:00:00','%Y/%m/%d %H:%M:%S') # simulation time output is number of hours since 00:00:00 01-01-1950
time_u54 = origin_time_u54 + datetime.timedelta(**{interval_type54: interval_num54})
print(time_u54)



In [ ]:
# ROMS
velocity_input_file_u54 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/ocean_velocities_u_ROMS_Jun2018-Dec2019_daily.nc'
velocity_input_file_v54 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/ocean_velocities_v_ROMS_Jun2018-Dec2019_daily.nc'

openfile_u54R = Dataset(velocity_input_file_u54)
openfile_v54R = Dataset(velocity_input_file_v54)
lon_vec_u54R = openfile_u54R.variables['lon_u']
lat_vec_u54R = openfile_u54R.variables['lat_u']
lon_vec_v54R = openfile_v54R.variables['lon_v']
lat_vec_v54R = openfile_v54R.variables['lat_v']
lon_grid_u54R = lon_vec_u54R
lat_grid_u54R = lat_vec_u54R
lon_grid_v54R = lon_vec_v54R
lat_grid_v54R = lat_vec_v54R
time_u54R = openfile_u54R.variables['ocean_time']
u54R =  openfile_u54R.variables['u']
v54R =  openfile_v54R.variables['v']

u_subset54R = u54R[53,0,:-1,:]
v_subset54R = v54R[53,0,:,:-1]
speed_subset54R = np.sqrt(np.abs(u_subset54R)**2 + np.abs(v_subset54R)**2)

interval_type54R = 'days'
interval_num54R = int(time_u54R[53])

origin_time_u54R = datetime.datetime.strptime('2018-06-01 12:00:00','%Y-%m-%d %H:%M:%S') # simulation time output is number of hours since 00:00:00 01-01-1950
time_u54R = origin_time_u54R + datetime.timedelta(**{interval_type54R: interval_num54R})
print(time_u54R)


In [ ]:
# wind input
wind_input_file_u = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/windage_u_avgd_interpolated_Jun2018-Oct2019_daily.nc'
wind_input_file_v = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/windage_v_avgd_interpolated_Jun2018-Oct2019_daily.nc'

openfile_windu = Dataset(wind_input_file_u)
lon_vec_windu = openfile_windu.variables['lon']
lat_vec_windu = openfile_windu.variables['lat']
time_windu = openfile_windu.variables['time']
windu =  openfile_windu.variables['__xarray_dataarray_variable__']
openfile_windv = Dataset(wind_input_file_v)
lon_vec_windv = openfile_windv.variables['lon']
lat_vec_windv = openfile_windv.variables['lat']
time_windv = openfile_windv.variables['time']
windv =  openfile_windv.variables['__xarray_dataarray_variable__']

windu_cut = windu[:124]
windv_cut = windv[:124]

average_subset_windu = np.mean(windu_cut, axis=0)
average_subset_windv = np.mean(windv_cut, axis=0)

# create grids of coordinates rather than just lists
lon_grid_windu, lat_grid_windu = np.meshgrid(lon_vec_windu, lat_vec_windu)
lon_grid_windv, lat_grid_windv = np.meshgrid(lon_vec_windv, lat_vec_windv)

# defining datetime objects for start and end time of simulation
interval_type = 'hours'
interval_num_start = int(time_windu[0])
interval_num_end = int(time_windu[-1])

origin_time_windu = datetime.datetime.strptime('1950/01/01 00:00:00','%Y/%m/%d %H:%M:%S') # simulation time output is number of hours since 00:00:00 01-01-1950
start_time_windu = origin_time_windu + datetime.timedelta(**{interval_type: interval_num_start}) # turns integers into a datetime object to be added to origin_time
end_time_windu = origin_time_windu + datetime.timedelta(**{interval_type: interval_num_end})


In [ ]:
# wind input
wind_input_file_upost = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/windage_u_avgd_interpolated_Oct2018-Oct2019_daily.nc'
wind_input_file_vpost = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/windage_v_avgd_interpolated_Oct2018-Oct2019_daily.nc'

openfile_windupost = Dataset(wind_input_file_upost)
lon_vec_windupost = openfile_windupost.variables['lon']
lat_vec_windupost = openfile_windupost.variables['lat']
time_windupost = openfile_windupost.variables['time']
windupost =  openfile_windupost.variables['__xarray_dataarray_variable__']
openfile_windvpost = Dataset(wind_input_file_vpost)
lon_vec_windvpost = openfile_windvpost.variables['lon']
lat_vec_windvpost = openfile_windvpost.variables['lat']
time_windvpost = openfile_windvpost.variables['time']
windvpost =  openfile_windvpost.variables['__xarray_dataarray_variable__']

windupost_cut = windupost[:124]
windvpost_cut = windvpost[:124]

average_subset_windupost = np.mean(windupost_cut, axis=0)
average_subset_windvpost = np.mean(windvpost_cut, axis=0)

# create grids of coordinates rather than just lists
lon_grid_windupost, lat_grid_windupost = np.meshgrid(lon_vec_windupost, lat_vec_windupost)
lon_grid_windvpost, lat_grid_windvpost = np.meshgrid(lon_vec_windvpost, lat_vec_windvpost)

# defining datetime objects for start and end time of simulation
interval_type = 'hours'
interval_num_start = int(time_windupost[0])
interval_num_end = int(time_windupost[-1])

origin_time_windupost = datetime.datetime.strptime('1950/01/01 00:00:00','%Y/%m/%d %H:%M:%S') # simulation time output is number of hours since 00:00:00 01-01-1950
start_time_windupost = origin_time_windupost + datetime.timedelta(**{interval_type: interval_num_start}) # turns integers into a datetime object to be added to origin_time
end_time_windupost = origin_time_windupost + datetime.timedelta(**{interval_type: interval_num_end})


In [ ]:
# wind input
wind_input_file_upre = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/windage_u_avgd_interpolated_Feb-Oct2019_daily.nc'
wind_input_file_vpre = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/windage_v_avgd_interpolated_Feb-Oct2019_daily.nc'

openfile_windupre = Dataset(wind_input_file_upre)
lon_vec_windupre = openfile_windupre.variables['lon']
lat_vec_windupre = openfile_windupre.variables['lat']
time_windupre = openfile_windupre.variables['time']
windupre =  openfile_windupre.variables['__xarray_dataarray_variable__']
openfile_windvpre = Dataset(wind_input_file_vpre)
lon_vec_windvpre = openfile_windvpre.variables['lon']
lat_vec_windvpre = openfile_windvpre.variables['lat']
time_windvpre = openfile_windvpre.variables['time']
windvpre =  openfile_windvpre.variables['__xarray_dataarray_variable__']

windupre_cut = windupre[:121]
windvpre_cut = windvpre[:121]

average_subset_windupre = np.mean(windupre_cut, axis=0)
average_subset_windvpre = np.mean(windvpre_cut, axis=0)

# create grids of coordinates rather than just lists
lon_grid_windupre, lat_grid_windupre = np.meshgrid(lon_vec_windupre, lat_vec_windupre)
lon_grid_windvpre, lat_grid_windvpre = np.meshgrid(lon_vec_windvpre, lat_vec_windvpre)

# defining datetime objects for start and end time of simulation
interval_type = 'hours'
interval_num_start = int(time_windupre[0])
interval_num_end = int(time_windupre[-1])

origin_time_windupre = datetime.datetime.strptime('1950/01/01 00:00:00','%Y/%m/%d %H:%M:%S') # simulation time output is number of hours since 00:00:00 01-01-1950
start_time_windupre = origin_time_windupre + datetime.timedelta(**{interval_type: interval_num_start}) # turns integers into a datetime object to be added to origin_time
end_time_windupre = origin_time_windupre + datetime.timedelta(**{interval_type: interval_num_end})


In [ ]:
# stokes input
stokes_input_file_u = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/stokes_u_avgd_interpolated_Jun2018-Oct2019_daily_fromcoarse.nc'
stokes_input_file_v = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/stokes_v_avgd_interpolated_Jun2018-Oct2019_daily_fromcoarse.nc'

openfile_stokesu = Dataset(stokes_input_file_u)
lon_vec_stokesu = openfile_stokesu.variables['lon']
lat_vec_stokesu = openfile_stokesu.variables['lat']
time_stokesu = openfile_stokesu.variables['time']
stokesu =  openfile_stokesu.variables['__xarray_dataarray_variable__']
openfile_stokesv = Dataset(stokes_input_file_v)
lon_vec_stokesv = openfile_stokesv.variables['lon']
lat_vec_stokesv = openfile_stokesv.variables['lat']
time_stokesv = openfile_stokesv.variables['time']
stokesv =  openfile_stokesv.variables['__xarray_dataarray_variable__']

stokesu_cut = stokesu[:123]
stokesv_cut = stokesv[:123]

average_subset_stokesu = np.mean(stokesu_cut, axis=0)
average_subset_stokesv = np.mean(stokesv_cut, axis=0)

# create grids of coordinates rather than just lists
lon_grid_stokesu, lat_grid_stokesu = np.meshgrid(lon_vec_stokesu, lat_vec_stokesu)
lon_grid_stokesv, lat_grid_stokesv = np.meshgrid(lon_vec_stokesv, lat_vec_stokesv)

# defining datetime objects for start and end time of simulation
interval_type = 'hours'
interval_num_start = int(time_stokesu[0])
interval_num_end = int(time_stokesu[-1])

origin_time_stokesu = datetime.datetime.strptime('1950/01/01 00:00:00','%Y/%m/%d %H:%M:%S') # simulation time output is number of hours since 00:00:00 01-01-1950
start_time_stokesu = origin_time_stokesu + datetime.timedelta(**{interval_type: interval_num_start}) # turns integers into a datetime object to be added to origin_time
end_time_stokesu = origin_time_stokesu + datetime.timedelta(**{interval_type: interval_num_end})


In [ ]:
# stokes input
stokes_input_file_upost = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/stokes_u_avgd_interpolated_Oct2018-Oct2019_daily_fromcoarse.nc'
stokes_input_file_vpost = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/stokes_v_avgd_interpolated_Oct2018-Oct2019_daily_fromcoarse.nc'

openfile_stokesupost = Dataset(stokes_input_file_upost)
lon_vec_stokesupost = openfile_stokesupost.variables['lon']
lat_vec_stokesupost = openfile_stokesupost.variables['lat']
time_stokesupost = openfile_stokesupost.variables['time']
stokesupost =  openfile_stokesupost.variables['__xarray_dataarray_variable__']
openfile_stokesvpost = Dataset(stokes_input_file_vpost)
lon_vec_stokesvpost = openfile_stokesvpost.variables['lon']
lat_vec_stokesvpost = openfile_stokesvpost.variables['lat']
time_stokesvpost = openfile_stokesvpost.variables['time']
stokesvpost =  openfile_stokesvpost.variables['__xarray_dataarray_variable__']

stokesupost_cut = stokesupost[:124]
stokesvpost_cut = stokesvpost[:124]

average_subset_stokesupost = np.mean(stokesupost_cut, axis=0)
average_subset_stokesvpost = np.mean(stokesvpost_cut, axis=0)

# create grids of coordinates rather than just lists
lon_grid_stokesupost, lat_grid_stokesupost = np.meshgrid(lon_vec_stokesupost, lat_vec_stokesupost)
lon_grid_stokesvpost, lat_grid_stokesvpost = np.meshgrid(lon_vec_stokesvpost, lat_vec_stokesvpost)

# defining datetime objects for start and end time of simulation
interval_type = 'hours'
interval_num_start = int(time_stokesupost[0])
interval_num_end = int(time_stokesupost[-1])

origin_time_stokesupost = datetime.datetime.strptime('1950/01/01 00:00:00','%Y/%m/%d %H:%M:%S') # simulation time output is number of hours since 00:00:00 01-01-1950
start_time_stokesupost = origin_time_stokesupost + datetime.timedelta(**{interval_type: interval_num_start}) # turns integers into a datetime object to be added to origin_time
end_time_stokesupost = origin_time_stokesupost + datetime.timedelta(**{interval_type: interval_num_end})


In [ ]:
# stokes input
stokes_input_file_upre = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/stokes_u_avgd_interpolated_Feb-Oct2019_daily_fromcoarse.nc'
stokes_input_file_vpre = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/stokes_v_avgd_interpolated_Feb-Oct2019_daily_fromcoarse.nc'

openfile_stokesupre = Dataset(stokes_input_file_upre)
lon_vec_stokesupre = openfile_stokesupre.variables['lon']
lat_vec_stokesupre = openfile_stokesupre.variables['lat']
time_stokesupre = openfile_stokesupre.variables['time']
stokesupre =  openfile_stokesupre.variables['__xarray_dataarray_variable__']
openfile_stokesvpre = Dataset(stokes_input_file_vpre)
lon_vec_stokesvpre = openfile_stokesvpre.variables['lon']
lat_vec_stokesvpre = openfile_stokesvpre.variables['lat']
time_stokesvpre = openfile_stokesvpre.variables['time']
stokesvpre =  openfile_stokesvpre.variables['__xarray_dataarray_variable__']

stokesupre_cut = stokesupre[:121]
stokesvpre_cut = stokesvpre[:121]

average_subset_stokesupre = np.mean(stokesupre_cut, axis=0)
average_subset_stokesvpre = np.mean(stokesvpre_cut, axis=0)

# create grids of coordinates rather than just lists
lon_grid_stokesupre, lat_grid_stokesupre = np.meshgrid(lon_vec_stokesupre, lat_vec_stokesupre)
lon_grid_stokesvpre, lat_grid_stokesvpre = np.meshgrid(lon_vec_stokesvpre, lat_vec_stokesvpre)

# defining datetime objects for start and end time of simulation
interval_type = 'hours'
interval_num_start = int(time_stokesupre[0])
interval_num_end = int(time_stokesupre[-1])

origin_time_stokesupre = datetime.datetime.strptime('1950/01/01 00:00:00','%Y/%m/%d %H:%M:%S') # simulation time output is number of hours since 00:00:00 01-01-1950
start_time_stokesupre = origin_time_stokesupre + datetime.timedelta(**{interval_type: interval_num_start}) # turns integers into a datetime object to be added to origin_time
end_time_stokesupre = origin_time_stokesupre + datetime.timedelta(**{interval_type: interval_num_end})


In [ ]:
avg_windu = np.mean(average_subset_windu)
avg_windupost = np.mean(average_subset_windupost)
avg_windupre = np.mean(average_subset_windupre)
avg_stokesu = np.mean(average_subset_stokesu)
avg_stokesupost = np.mean(average_subset_stokesupost)
avg_stokesupre = np.mean(average_subset_stokesupre)

avg_windv = np.mean(average_subset_windv)
avg_windvpost = np.mean(average_subset_windvpost)
avg_windvpre = np.mean(average_subset_windvpre)
avg_stokesv = np.mean(average_subset_stokesv)
avg_stokesvpost = np.mean(average_subset_stokesvpost)
avg_stokesvpre = np.mean(average_subset_stokesvpre)

avg_winduboth = (avg_windupre+avg_windu)/2
avg_windvboth = (avg_windvpre+avg_windv)/2
avg_stokesuboth = (avg_stokesupre+avg_stokesu)/2
avg_stokesvboth = (avg_stokesvpre+avg_stokesv)/2


In [ ]:
# post-monsoon

# create figure
fig = plt.figure(figsize=(20, 10), dpi=300)
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([77,99,3.5,24]) # full domain
# ax.add_feature(cartopy.feature.OCEAN)
# ax.add_feature(cartopy.feature.LAND, facecolor='white')
ax.add_feature(cartopy.feature.BORDERS, linestyle='-', linewidth=1.0)
# ax.coastlines()
gl = ax.gridlines(draw_labels=True)
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}

transform = ccrs.PlateCarree()._as_mpl_transform(ax)

m1 = ax.pcolormesh(lon_grid_u54R,lat_grid_u54R,speed_subset54R, cmap='Blues_r', vmin=np.amin(speed_subset54R), vmax=np.amax(speed_subset54R)) # mappable content

ax.scatter(lon_p_SL[0:,0], lat_p_SL[0:,0],color='lightpink', s=10)
ax.scatter(lon_p_India[0:,0], lat_p_India[0:,0],color='gold', s=10)
ax.scatter(lon_p_Bang[0:,0], lat_p_Bang[0:,0],color='crimson', s=10)
ax.scatter(lon_p_Myan[0:,0], lat_p_Myan[0:,0],color='mediumaquamarine', s=10)
ax.scatter(lon_p_Thai[0:,0], lat_p_Thai[0:,0],color='black', s=10)
ax.scatter(lon_p_Indo[0:,0], lat_p_Indo[0:,0],color='cyan', s=10)

ax.annotate('Sri', xy=(80.5, 7.7), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('Lanka', xy=(80.1, 7.1), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('India', xy=(78.7, 17), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('Bangladesh', xy=(88.6, 23.6), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('Myanmar', xy=(95, 21), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('Thailand', xy=(97.0, 9.1), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('Indonesia', xy=(96, 4.3), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('A&N', xy=(92, 10), xycoords=transform,
            fontsize=12, color='k')

ax.annotate('EICC', xy=(82, 14), xycoords=transform, rotation=60,
            fontsize=12, color='k')
ax.annotate('NMC', xy=(87, 5), xycoords=transform,
            fontsize=12, color='k')

x_pos_wind_post = 85.2
y_pos_wind_post = 21.1
plt.quiver(x_pos_wind_post, y_pos_wind_post, avg_windupost, avg_windvpost, scale=1.2, scale_units='xy', angles='xy')
ax.annotate('Wind', xy=(82.8, 20.5), xycoords=transform,
            fontsize=12, color='k')

x_pos_stokes_post = 85.2
y_pos_stokes_post = 23.1
plt.quiver(x_pos_stokes_post, y_pos_stokes_post, avg_stokesupost, avg_stokesvpost, scale=0.01, scale_units='xy', angles='xy')
ax.annotate('Stokes', xy=(82.4, 22.5), xycoords=transform,
            fontsize=12, color='k')

fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/BoB_template_plain_postmonsoon", bbox_inches='tight', facecolor='white', transparent=False)


In [ ]:
# pre/monsoon
# create figure
fig = plt.figure(figsize=(20, 10), dpi=300)
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([77,99,3.5,24]) # full domain
# ax.add_feature(cartopy.feature.OCEAN)
# ax.add_feature(cartopy.feature.LAND, facecolor='white')
ax.add_feature(cartopy.feature.BORDERS, linestyle='-', linewidth=1.0)
# ax.coastlines()
gl = ax.gridlines(draw_labels=True)
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}

transform = ccrs.PlateCarree()._as_mpl_transform(ax)

m1 = ax.pcolormesh(lon_grid_u54,lat_grid_u54,speed_subset54, cmap='Blues_r', vmin=np.amin(speed_subset54), vmax=np.amax(speed_subset54)) # mappable content

p0 = ax.scatter(lon_p_SL[0:,0], lat_p_SL[0:,0],color='lightpink', s=10)
p1 = ax.scatter(lon_p_India[0:,0], lat_p_India[0:,0],color='gold', s=10)
p2 = ax.scatter(lon_p_Bang[0:,0], lat_p_Bang[0:,0],color='crimson', s=10)
p3 = ax.scatter(lon_p_Myan[0:,0], lat_p_Myan[0:,0],color='mediumaquamarine', s=10)
p4 = ax.scatter(lon_p_Thai[0:,0], lat_p_Thai[0:,0],color='black', s=10)
p5 = ax.scatter(lon_p_Indo[0:,0], lat_p_Indo[0:,0],color='cyan', s=10)

ax.annotate('Sri', xy=(80.5, 7.7), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('Lanka', xy=(80.1, 7.1), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('India', xy=(78.7, 17), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('Bangladesh', xy=(88.6, 23.6), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('Myanmar', xy=(95, 21), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('Thailand', xy=(97.0, 9.1), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('Indonesia', xy=(96, 4.3), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('A&N', xy=(92, 10), xycoords=transform,
            fontsize=12, color='k')

ax.annotate('EICC', xy=(82, 14), xycoords=transform, rotation=60,
            fontsize=12, color='k')
ax.annotate('SMC', xy=(87, 5), xycoords=transform,
            fontsize=12, color='k')
ax.annotate('SLD', xy=(85, 7.4), xycoords=transform,
            fontsize=12, color='k')

plt.legend((p0, p1, p2, p3, p4, p5),
           ('Sri Lanka', 'India', 'Bangladesh', 'Myanmar', 'Thailand', 'Indonesia'),
           scatterpoints=5,
           loc='upper left',
           ncol=1,
           fontsize=12)

x_pos_wind = 84
y_pos_wind = 20
plt.quiver(x_pos_wind, y_pos_wind, avg_winduboth, avg_windvboth, scale=1.5, scale_units='xy', angles='xy')
ax.annotate('Wind', xy=(82.8, 20.5), xycoords=transform,
            fontsize=12, color='k')

x_pos_stokes = 84
y_pos_stokes = 22
plt.quiver(x_pos_stokes, y_pos_stokes, avg_stokesuboth, avg_stokesvboth, scale=0.02, scale_units='xy', angles='xy')
ax.annotate('Stokes', xy=(82.4, 22.5), xycoords=transform,
            fontsize=12, color='k')

fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/BoB_template_plain_premonsoon", bbox_inches='tight', facecolor='white', transparent=False)


In [ ]:
# post-monsoon zoom in

# create figure
fig = plt.figure(figsize=(20, 10), dpi=300)
ax = plt.axes(projection=ccrs.PlateCarree())
# ax.set_extent([77,99,3.5,24]) # full domain
ax.set_extent([93,99,3.5,9])
ax.add_feature(cartopy.feature.BORDERS, linestyle='-', linewidth=1.0)
gl = ax.gridlines(draw_labels=True)
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}

transform = ccrs.PlateCarree()._as_mpl_transform(ax)

m1 = ax.pcolormesh(lon_grid_u54R,lat_grid_u54R,speed_subset54R, cmap='Blues_r', vmin=np.amin(speed_subset54R), vmax=np.amax(speed_subset54R)) # mappable content

ax.scatter(lon_p_Thai[0:,0], lat_p_Thai[0:,0],color='black', s=50)
ax.scatter(lon_p_Indo[0:,0], lat_p_Indo[0:,0],color='cyan', s=50)

cbar = fig.colorbar(m1, ax=ax,location='bottom', cmap='Blues')

# Get the current position of the colorbar
pos = cbar.ax.get_position()

# Set the new position of the colorbar
cbar.ax.set_position([pos.x0+0.25, pos.y0+0.075, pos.width-0.5, pos.height])
cbar.set_label('m/s', fontsize=12, x=1.04, labelpad=-32)

fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/BoB_ROMS_start_post_zoomed_indo", bbox_inches='tight', facecolor='white', transparent=False)


In [ ]:
# pre/monsoon
# create figure
fig = plt.figure(figsize=(20, 10), dpi=300)
ax = plt.axes(projection=ccrs.PlateCarree())
# ax.set_extent([77,99,3.5,24]) # full domain
ax.set_extent([93,99,3.5,9])
ax.add_feature(cartopy.feature.BORDERS, linestyle='-', linewidth=1.0)
gl = ax.gridlines(draw_labels=True)
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}

transform = ccrs.PlateCarree()._as_mpl_transform(ax)

m1 = ax.pcolormesh(lon_grid_u54,lat_grid_u54,speed_subset54, cmap='Blues_r', vmin=np.amin(speed_subset54), vmax=np.amax(speed_subset54)) # mappable content

p4 = ax.scatter(lon_p_Thai[0:,0], lat_p_Thai[0:,0],color='black', s=50)
p5 = ax.scatter(lon_p_Indo[0:,0], lat_p_Indo[0:,0],color='cyan', s=50)

cbar = fig.colorbar(m1, ax=ax,location='bottom', cmap='Blues')

# Get the current position of the colorbar
pos = cbar.ax.get_position()

# Set the new position of the colorbar
cbar.ax.set_position([pos.x0+0.25, pos.y0+0.075, pos.width-0.5, pos.height])
cbar.set_label('m/s', fontsize=12, x=1.04, labelpad=-32)

fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/BoB_CMEMS_start_pre_zoomed_indo", bbox_inches='tight', facecolor='white', transparent=False)


In [ ]:
# Figure 2

In [ ]:
drifter_data = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/misc/drifter_6hour_undrogued_only.csv'
cols = ["WMO", "time", "latitiude", "longitude"]
data_df = pd.read_csv(drifter_data, sep=",", skipinitialspace=True, header = 1, names=cols)

In [ ]:
lat_drifter1 = data_df.latitiude[data_df.WMO==1401521] 
lon_drifter1 = data_df.longitude[data_df.WMO==1401521]
lat_drifter2 = data_df.latitiude[data_df.WMO==5301513] 
lon_drifter2 = data_df.longitude[data_df.WMO==5301513]
lat_drifter3 = data_df.latitiude[data_df.WMO==1101522]  
lon_drifter3 = data_df.longitude[data_df.WMO==1101522]
lat_drifter4 = data_df.latitiude[data_df.WMO==4101590] # yellow = D1
lon_drifter4 = data_df.longitude[data_df.WMO==4101590]
lat_drifter5 = data_df.latitiude[data_df.WMO==2301617] # red = D2
lon_drifter5 = data_df.longitude[data_df.WMO==2301617]
lat_drifter6 = data_df.latitiude[data_df.WMO==2301619] # pink = D3
lon_drifter6 = data_df.longitude[data_df.WMO==2301619]
lat_drifter7 = data_df.latitiude[data_df.WMO==2301604] # green = D4
lon_drifter7 = data_df.longitude[data_df.WMO==2301604]
lat_drifter8 = data_df.latitiude[data_df.WMO==5301585] # orange = D5
lon_drifter8 = data_df.longitude[data_df.WMO==5301585]

In [ ]:
particle_input_file_d1 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/validation/BoB_Cop_daily_validation_yellow.nc'
particle_input_fileR_d1 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/validation/BoB_ROMS_daily_validation_yellow.nc'
particle_input_file_d2 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/validation/BoB_Cop_daily_validation_red.nc'
particle_input_fileR_d2 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/validation/BoB_ROMS_daily_validation_red.nc'
particle_input_file_d3 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/validation/BoB_Cop_daily_validation_pink.nc'
particle_input_fileR_d3 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/validation/BoB_ROMS_daily_validation_pink.nc'
particle_input_file_d4 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/validation/BoB_Cop_daily_validation_green.nc'
particle_input_fileR_d4 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/validation/BoB_ROMS_daily_validation_green.nc'
particle_input_file_d5 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/validation/BoB_Cop_daily_validation_orange.nc'
particle_input_fileR_d5 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/validation/BoB_ROMS_daily_validation_orange.nc'

# import data
openfile_p1 = Dataset(particle_input_file_d1)
lon_p1 = openfile_p1.variables['lon']
lat_p1 = openfile_p1.variables['lat']
time_p1 = openfile_p1.variables['time']

openfile_pR1 = Dataset(particle_input_fileR_d1)
lon_pR1 = openfile_pR1.variables['lon']
lat_pR1 = openfile_pR1.variables['lat']
time_pR1 = openfile_pR1.variables['time']

# import data
openfile_p2 = Dataset(particle_input_file_d2)
lon_p2 = openfile_p2.variables['lon']
lat_p2 = openfile_p2.variables['lat']
time_p2 = openfile_p2.variables['time']

openfile_pR2 = Dataset(particle_input_fileR_d2)
lon_pR2 = openfile_pR2.variables['lon']
lat_pR2 = openfile_pR2.variables['lat']
time_pR2 = openfile_pR2.variables['time']

# import data
openfile_p3 = Dataset(particle_input_file_d3)
lon_p3 = openfile_p3.variables['lon']
lat_p3 = openfile_p3.variables['lat']
time_p3 = openfile_p3.variables['time']

openfile_pR3 = Dataset(particle_input_fileR_d3)
lon_pR3 = openfile_pR3.variables['lon']
lat_pR3 = openfile_pR3.variables['lat']
time_pR3 = openfile_pR3.variables['time']

# import data
openfile_p4 = Dataset(particle_input_file_d4)
lon_p4 = openfile_p4.variables['lon']
lat_p4 = openfile_p4.variables['lat']
time_p4 = openfile_p4.variables['time']

openfile_pR4 = Dataset(particle_input_fileR_d4)
lon_pR4 = openfile_pR4.variables['lon']
lat_pR4 = openfile_pR4.variables['lat']
time_pR4 = openfile_pR4.variables['time']

# import data
openfile_p5 = Dataset(particle_input_file_d5)
lon_p5 = openfile_p5.variables['lon']
lat_p5 = openfile_p5.variables['lat']
time_p5 = openfile_p5.variables['time']

openfile_pR5 = Dataset(particle_input_fileR_d5)
lon_pR5 = openfile_pR5.variables['lon']
lat_pR5 = openfile_pR5.variables['lat']
time_pR5 = openfile_pR5.variables['time']

In [ ]:
# Copernicus input
velocity_input_file_cyan = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/validation/ocean_velocities_Cop_Jun-July2019_daily_validation_orange.nc'

openfile_u_cyan = Dataset(velocity_input_file_cyan)
lon_vec_u_cyan = openfile_u_cyan.variables['longitude']
lat_vec_u_cyan = openfile_u_cyan.variables['latitude']
time_u_cyan = openfile_u_cyan.variables['time']
u_cyan =  openfile_u_cyan.variables['uo']
v_cyan =  openfile_u_cyan.variables['vo']

u_subset_cyan= u_cyan[0,0,:,:]
v_subset_cyan= v_cyan[0,0,:,:]
speed_subset_cyan = np.sqrt(np.abs(u_subset_cyan)**2 + np.abs(v_subset_cyan)**2)

# create grids of coordinates rather than just lists
lon_grid_u, lat_grid_u = np.meshgrid(lon_vec_u_cyan, lat_vec_u_cyan)

# defining datetime objects for start and end time of simulation
interval_type = 'hours'
interval_num_start_cyan = int(time_u_cyan[0])
interval_num_end_cyan = int(time_u_cyan[-1])

origin_time_u_cyan = datetime.datetime.strptime('1950/01/01 00:00:00','%Y/%m/%d %H:%M:%S') # simulation time output is number of hours since 00:00:00 01-01-1950
start_time_u_cyan = origin_time_u_cyan + datetime.timedelta(**{interval_type: interval_num_start_cyan}) # turns integers into a datetime object to be added to origin_time
end_time_u_cyan = origin_time_u_cyan + datetime.timedelta(**{interval_type: interval_num_end_cyan})
print(start_time_u_cyan)


In [ ]:
# ROMS input
velocity_input_file_u_cyan = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/validation/ocean_velocities_u_ROMS_Jun-July2019_daily_validation_orange.nc'
velocity_input_file_v_cyan = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/validation/ocean_velocities_v_ROMS_Jun-July2019_daily_validation_orange.nc'

openfile_uR_cyan = Dataset(velocity_input_file_u_cyan)
openfile_vR_cyan = Dataset(velocity_input_file_v_cyan)
lon_vec_uR_cyan = openfile_uR_cyan.variables['lon_u']
lat_vec_uR_cyan = openfile_uR_cyan.variables['lat_u']
lon_vec_vR_cyan = openfile_vR_cyan.variables['lon_v']
lat_vec_vR_cyan = openfile_vR_cyan.variables['lat_v']
time_uR_cyan = openfile_uR_cyan.variables['ocean_time']
uR_cyan =  openfile_uR_cyan.variables['u']
vR_cyan =  openfile_vR_cyan.variables['v']

u_subsetR_cyan = uR_cyan[0,0,:-1,:]
v_subsetR_cyan = vR_cyan[0,0,:,:-1]
speed_subsetR_cyan = np.sqrt(np.abs(u_subsetR_cyan)**2 + np.abs(v_subsetR_cyan)**2)

# # create grids of coordinates rather than just lists
lon_grid_uR = lon_vec_uR_cyan
lat_grid_uR = lat_vec_uR_cyan

# defining datetime objects for start and end time of simulation
interval_type = 'days'
interval_num_startR_cyan = int(time_uR_cyan[0])
interval_num_endR_cyan = int(time_uR_cyan[-1])

origin_time_uR_cyan = datetime.datetime.strptime('2018-06-01 12:00:00','%Y-%m-%d %H:%M:%S') # simulation time output is number of hours since 00:00:00 01-01-1950
start_time_uR_cyan = origin_time_uR_cyan + datetime.timedelta(**{interval_type: interval_num_startR_cyan}) # turns integers into a datetime object to be added to origin_time
end_time_uR_cyan = origin_time_uR_cyan + datetime.timedelta(**{interval_type: interval_num_endR_cyan})
print(start_time_uR_cyan)


In [ ]:
CMEMS_sep_dists = [55.76250788,59.79824339,70.3053312,70.62065673,37.90227422]
CMEMS_std_devs = [37.09595994,28.19078151,38.90275448,38.02558416,17.43446894]
ROMS_sep_dists = [43.65731458,79.63193581,143.5152277,91.20180301,75.34572541]
ROMS_std_devs = [23.22829011,63.825822,77.01788581,49.5593881,34.39362725]
 
colours = ['lightpink', 'gold', 'crimson', 'mediumaquamarine', 'cyan']
drifters = ['D1', 'D2', 'D3', 'D4', 'D5']

In [ ]:
# Create a figure and GridSpec
fig = plt.figure(figsize=(18, 15), dpi=300)
gs = GridSpec(2, 3)

# Create a bigger subplot on the left
# all drifter tracks figure
ax1 = fig.add_subplot(gs[:, 0], projection=ccrs.PlateCarree())
ax1.set_extent([77, 99, 3.5, 24])  # BoB domain
# ax1.coastlines()
gl = ax1.gridlines(draw_labels=True)
gl.xlabel_style = {'fontsize': 16}
gl.ylabel_style = {'fontsize': 16}

m1 = ax1.pcolormesh(lon_grid_uR,lat_grid_uR,speed_subsetR_cyan, cmap='Blues_r', vmin=np.amin(speed_subsetR_cyan), vmax=np.amax(speed_subsetR_cyan)) # mappable content

cbar = fig.colorbar(m1, location='bottom')
cbar.set_label('m/s', fontsize=16, x=1.045, labelpad=-44)
cbar.ax.tick_params(labelsize=16)
pos = cbar.ax.get_position()
cbar.ax.set_position([pos.x0, pos.y0, pos.width-0.4, pos.height])

p1 = ax1.scatter(lon_drifter4, lat_drifter4, transform=ccrs.PlateCarree(), color='lightpink', s=10)
p2 = ax1.scatter(lon_drifter5, lat_drifter5, transform=ccrs.PlateCarree(), color='gold', s=10)
p3 = ax1.scatter(lon_drifter6, lat_drifter6, transform=ccrs.PlateCarree(), color='crimson', s=10)
p4 = ax1.scatter(lon_drifter7, lat_drifter7, transform=ccrs.PlateCarree(), color='mediumaquamarine', s=10)
p5 = ax1.scatter(lon_drifter8, lat_drifter8, transform=ccrs.PlateCarree(), color='cyan', s=10)
ax1.scatter(lon_drifter4[0:1], lat_drifter4[0:1], transform=ccrs.PlateCarree(), color='lightpink', s=500, marker='*')
ax1.scatter(lon_drifter5[0:1], lat_drifter5[0:1], transform=ccrs.PlateCarree(), color='gold', s=500, marker='*')
ax1.scatter(lon_drifter6[0:1], lat_drifter6[0:1], transform=ccrs.PlateCarree(), color='crimson', s=500, marker='*')
ax1.scatter(lon_drifter7[0:1], lat_drifter7[0:1], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=500, marker='*')
ax1.scatter(lon_drifter8[0:1], lat_drifter8[0:1], transform=ccrs.PlateCarree(), color='cyan', s=500, marker='*')

plt.legend((p1, p2, p3, p4, p5),
           ('D1: 10 July - 1 Aug 2018', 'D2: 21 Sept - 4 Dec 2018', 'D3: 27 Aug - 22 Oct 2018', 'D4: 18 Sept 2018 - 2 Aug 2019', 'D5: 2 Jun - 11 July 2019'),
           scatterpoints=5,
           loc='upper left',
           ncol=1,
           fontsize=16)


# Create two smaller subplots on the right
ax2 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
ax2.set_extent([85,94,9,15]) # orange
ax2.coastlines()
gl2 = ax2.gridlines(draw_labels=True)
gl2.xlabel_style = {'fontsize': 16}
gl2.ylabel_style = {'fontsize': 16}

m2 = ax2.pcolormesh(lon_grid_uR,lat_grid_uR,speed_subsetR_cyan, cmap='Blues_r', vmin=np.amin(speed_subsetR_cyan), vmax=np.amax(speed_subsetR_cyan)) # mappable content

for ii in range(0,lon_pR5.shape[0],6):
    if 0 <= time_pR5[ii,0] < 604800:
        ax2.scatter(lon_pR5[ii,:8], lat_pR5[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=30)
for ii in range(1,lon_pR5.shape[0],6):
    if 604800 <= time_pR5[ii,0] < 1209600:
        ax2.scatter(lon_pR5[ii,:8], lat_pR5[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=30)
for ii in range(2,lon_pR5.shape[0],6):
    if 1209600 <= time_pR5[ii,0] < 1814400:
        ax2.scatter(lon_pR5[ii,:8], lat_pR5[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=30)
for ii in range(3,lon_pR5.shape[0],6):
    if 1814400 <= time_pR5[ii,0] < 2419200:
        ax2.scatter(lon_pR5[ii,:8], lat_pR5[ii,:8], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=30)
for ii in range(4,lon_pR5.shape[0],6):
    if 2419200 <= time_pR5[ii,0] < 3024000:
        ax2.scatter(lon_pR5[ii,:8], lat_pR5[ii,:8], transform=ccrs.PlateCarree(), color='k', s=30)

p6 = ax2.scatter(lon_drifter8[2:], lat_drifter8[2:], transform=ccrs.PlateCarree(), color='cyan', s=10)
ax2.scatter(lon_drifter8[0:1], lat_drifter8[0:1], transform=ccrs.PlateCarree(), color='cyan', s=300, marker='*')


ax3 = fig.add_subplot(gs[1, 1])

plt.rcParams['hatch.linewidth'] = 4.0

# Plot CMEMS data
bar_width = 0.35
index = np.arange(len(drifters))
for i in range(len(drifters)):
    ax3.bar(index[i] - bar_width/2, CMEMS_sep_dists[i], bar_width, color=colours[i], label=f'CMEMS {drifters[i]}', edgecolor='white')
    ax3.bar(index[i] + bar_width/2, ROMS_sep_dists[i], bar_width, color=colours[i], label=f'ROMS {drifters[i]}', edgecolor='white', hatch='//')#, alpha = 0.5, hatch='//')
    
# Add error bars
ax3.errorbar(index - bar_width/2, CMEMS_sep_dists, yerr=CMEMS_std_devs, fmt='none', color='black', capsize=5)
ax3.errorbar(index + bar_width/2, ROMS_sep_dists, yerr=ROMS_std_devs, fmt='none', color='black', capsize=5)

# Customize the plot
ax3.set_xticks(index)
ax3.set_xticklabels(drifters)
ax3.set_xlabel('Drifter', fontsize=16)
ax3.set_ylabel('Mean Cumulative Separation Distance (km)', fontsize=16)
ax3.tick_params(axis='both', which='major', labelsize=16)  # Larger tick labels

# Customize the legend
legend_labels = {'CMEMS': 'CMEMS', 'ROMS': 'ROMS'}
legend_elements = [
    patches.Rectangle((0, 0), 1, 1, facecolor='black', edgecolor='white', label=legend_labels['CMEMS']),
    patches.Rectangle((0, 0), 1, 1, facecolor='black', edgecolor='white', hatch='//', label=legend_labels['ROMS'])
]

ax3.legend(handles=legend_elements, fontsize=16)

ax1.set_position([0, 0, 0.63, 0.63])  # [left, bottom, width, height]
ax2.set_position([0.68, 0.295, 0.4, 0.35])  # [left, bottom, width, height]
ax3.set_position([0.69, -0.05, 0.42, 0.31])  # [left, bottom, width, height]

cbar.ax.set_position([pos.x0-0.05, pos.y0-0.26, pos.width+0.26, pos.height])

ax1.text(0.94, 0.99, "(a)", transform=ax1.transAxes, fontsize=20, fontweight='bold', va='top')
ax2.text(0.02, 0.98, "(b)", transform=ax2.transAxes, fontsize=20, fontweight='bold', va='top')
ax3.text(0.02, 0.97, "(c)", transform=ax3.transAxes, fontsize=20, fontweight='bold', va='top')

plt.tight_layout()

fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/fig02", bbox_inches='tight', facecolor='white', transparent=False)


In [ ]:
# Figure 3

In [ ]:
def plot_conn_mat_paper_noextras(ax, input_conn_mat_file):
    conn_mat_df = pd.read_csv(input_conn_mat_file, index_col=0)
    conn_mat_df.replace(0.0, np.nan, inplace=True) # replace all zeros with NaNs, inplace=True changes it in the dataframe itself not in the variable created here.
    polygon_names_list_source = list(conn_mat_df) # need to remove Andaman and Nicobar
    polygon_names_list_source = [polygon_names_list_source[x] for x in [0,1,2,3,4,5] ]
    polygon_names_list_sink = list(conn_mat_df) # this prints out the column headings of the dataframe. For connectivity marices, the column and row headings are the same so I don't need to extract the row headings (which I think is more complicated...). I can just use the column headings as xticklabels and yticklabels

    m1 = ax.matshow(conn_mat_df, cmap='Blues', vmin=0, vmax=1) # mappable content

    for (ii, jj), z in np.ndenumerate(conn_mat_df):
        if ~np.isnan(z):
            ax.text(jj, ii, '{:0.2f}'.format(z), ha='center', va='center')

    # Show all ticks and label them with the respective list entries
    ax.set_xticks(np.arange(len(polygon_names_list_sink)), labels=polygon_names_list_sink)
    ax.set_yticks(np.arange(len(polygon_names_list_source)), labels=polygon_names_list_source)
    ax.xaxis.set_ticks_position('bottom')
    # Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
         rotation_mode="anchor")

    if ax == axs[0,0]:
        ax.set_xticks([])
        ax.set_xticklabels([])
        ax.set_ylabel('Source', fontsize='x-large')
    elif ax == axs[0,1] or ax == axs[1,1] or ax == axs[2,1]:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])
    elif ax == axs[1,0] or ax == axs[2,0]:
        ax.set_xticks([])
        ax.set_xticklabels([])
        ax.set_ylabel('Source', fontsize='x-large')
    elif  ax == axs[3,0]:
        ax.set_xlabel('Sink', fontsize='x-large')
        ax.set_ylabel('Source', fontsize='x-large')
    elif  ax == axs[3,1]:
        ax.set_xlabel('Sink', fontsize='x-large') 
        ax.set_yticks([])
        ax.set_yticklabels([])
    

In [ ]:
input_conn_mat_file_year_CMEMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_year_Cop.csv'
input_conn_mat_file_year_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_year_ROMS.csv'
input_conn_mat_file_monsoon_CMEMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_Cop_daily_Jun2018-Sept2019_monsoon.csv'
input_conn_mat_file_monsoon_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_ROMS_daily_Jun2018-Sept2019_monsoon.csv'
input_conn_mat_file_post_CMEMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_Cop_daily_Oct2018-Sept2019_postmonsoon.csv'
input_conn_mat_file_post_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_ROMS_daily_Oct2018-Sept2019_postmonsoon.csv'
input_conn_mat_file_pre_CMEMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_Cop_daily_Feb-Sept2019_premonsoon.csv'
input_conn_mat_file_pre_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_ROMS_daily_Feb-Sept2019_premonsoon.csv'

In [ ]:
# Create a figure and axis objects with 4 rows and 2 columns
fig, axs = plt.subplots(4, 2, figsize=(8, 13), dpi=300)

# Add a title for each row
row_titles = ["Full simulation (Jun '18-Sept '19)", 'Monsoon (Jun-Sept)', 'Post-monsoon (Oct-Jan)', 'Pre-monsoon (Feb-May)']
for ii, title in enumerate(row_titles):
    # Position the text outside of the subfigure on the right side
    axs[ii, 0].text(1.04, 1.05, title, ha='center', va='center', fontsize=14, transform=axs[ii, 0].transAxes)
    
m = plot_conn_mat_paper_noextras(axs[0,0], input_conn_mat_file_year_CMEMS)
plot_conn_mat_paper_noextras(axs[0,1], input_conn_mat_file_year_ROMS)
plot_conn_mat_paper_noextras(axs[1,0], input_conn_mat_file_monsoon_CMEMS)
plot_conn_mat_paper_noextras(axs[1,1], input_conn_mat_file_monsoon_ROMS)
plot_conn_mat_paper_noextras(axs[2,0], input_conn_mat_file_post_CMEMS)
plot_conn_mat_paper_noextras(axs[2,1], input_conn_mat_file_post_ROMS)
plot_conn_mat_paper_noextras(axs[3,0], input_conn_mat_file_pre_CMEMS)
plot_conn_mat_paper_noextras(axs[3,1], input_conn_mat_file_pre_ROMS)

plt.subplots_adjust(left=0.1, right=0.9, bottom=0.2, top=0.9, wspace=0, hspace=None)

cbar = fig.colorbar(m, ax=axs[-1, :],location='bottom', cmap='Blues')

# Get the current position of the colorbar
pos = cbar.ax.get_position()

# Set the new position of the colorbar
cbar.ax.set_position([pos.x0-0.07, pos.y0-0.27, pos.width+0.2, pos.height])

axs[0,0].text(0, 1.07, "(a)", transform=axs[0,0].transAxes, fontsize=12, fontweight='bold', va='top')
axs[0,1].text(0.92, 1.07, "(b)", transform=axs[0,1].transAxes, fontsize=12, fontweight='bold', va='top')
axs[1,0].text(0, 1.07, "(c)", transform=axs[1,0].transAxes, fontsize=12, fontweight='bold', va='top')
axs[1,1].text(0.92, 1.07, "(d)", transform=axs[1,1].transAxes, fontsize=12, fontweight='bold', va='top')
axs[2,0].text(0, 1.07, "(e)", transform=axs[2,0].transAxes, fontsize=12, fontweight='bold', va='top')
axs[2,1].text(0.92, 1.07, "(f)", transform=axs[2,1].transAxes, fontsize=12, fontweight='bold', va='top')
axs[3,0].text(0, 1.07, "(g)", transform=axs[3,0].transAxes, fontsize=12, fontweight='bold', va='top')
axs[3,1].text(0.92, 1.07, "(h)", transform=axs[3,1].transAxes, fontsize=12, fontweight='bold', va='top')

axs[0,0].text(0.5, 1.16, "CMEMS", transform=axs[0,0].transAxes, fontsize=12, ha='center', va='top')
axs[0,1].text(0.5, 1.16, "ROMS", transform=axs[0,1].transAxes, fontsize=12, ha='center', va='top')

# Adjust spacing between subplots
plt.tight_layout()

fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/fig03", bbox_inches='tight', facecolor='white', transparent=False)


In [ ]:
# Figure 4

In [ ]:
def plot_conn_mat_paper_suppmat_noextras(ax, input_conn_mat_file):
    conn_mat_df = pd.read_csv(input_conn_mat_file, index_col=0)
    conn_mat_df.replace(0.0, np.nan, inplace=True) # replace all zeros with NaNs, inplace=True changes it in the dataframe itself not in the variable created here.
    polygon_names_list_source = list(conn_mat_df) # need to remove Andaman and Nicobar
    polygon_names_list_source = [polygon_names_list_source[x] for x in [0,1,2,3,4,5] ]
    polygon_names_list_sink = list(conn_mat_df) # this prints out the column headings of the dataframe. For connectivity marices, the column and row headings are the same so I don't need to extract the row headings (which I think is more complicated...). I can just use the column headings as xticklabels and yticklabels

    m1 = ax.matshow(conn_mat_df, cmap='Blues', vmin=0, vmax=1) # mappable content

    for (ii, jj), z in np.ndenumerate(conn_mat_df):
        if ~np.isnan(z):
            ax.text(jj, ii, '{:0.2f}'.format(z), ha='center', va='center')

    # Show all ticks and label them with the respective list entries
    ax.set_xticks(np.arange(len(polygon_names_list_sink)), labels=polygon_names_list_sink)
    ax.set_yticks(np.arange(len(polygon_names_list_source)), labels=polygon_names_list_source)
    ax.xaxis.set_ticks_position('bottom')
    # Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
         rotation_mode="anchor")

    if ax == axs[1,0]:
        ax.set_ylabel('Source', fontsize='x-large')
        ax.set_xticks([])
        ax.set_xticklabels([])
    elif ax == axs[2,1]:
        ax.set_xlabel('Sink', fontsize='x-large') 
        ax.set_yticks([])
        ax.set_yticklabels([])
    elif ax == axs[2,0]:
        ax.set_xlabel('Sink', fontsize='x-large')   
        ax.set_ylabel('Source', fontsize='x-large')
    elif ax == axs[1,1]:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])
    elif ax == axs[0,1]:
        ax.set_yticks([])
        ax.set_yticklabels([])
        ax.set_xticks([])
        ax.set_xticklabels([])
    elif ax == axs[0,0]:
        ax.set_xticks([])
        ax.set_xticklabels([])
        ax.set_ylabel('Source', fontsize='x-large')
    

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(8, 10), dpi=300)

# Add a title for each row
row_titles = ['Monsoon (Jun-Sept)', 'Post-monsoon (Oct-Jan)', 'Pre-monsoon (Feb-May)']
for ii, title in enumerate(row_titles):
    # Position the text outside of the subfigure on the right side
    axs[ii, 0].text(1.04, 1.05, title, ha='center', va='center', fontsize=14, transform=axs[ii, 0].transAxes)

m = plot_conn_mat_paper_suppmat_noextras(axs[0,0], input_conn_mat_file_monsoon_CMEMS)
plot_conn_mat_paper_suppmat_noextras(axs[0,1], input_conn_mat_file_monsoon_ROMS)
plot_conn_mat_paper_suppmat_noextras(axs[1,0], input_conn_mat_file_post_CMEMS)
plot_conn_mat_paper_suppmat_noextras(axs[1,1], input_conn_mat_file_post_ROMS)
plot_conn_mat_paper_suppmat_noextras(axs[2,0], input_conn_mat_file_pre_ROMS)
plot_conn_mat_paper_suppmat_noextras(axs[2,1], input_conn_mat_file_pre_ROMS)

cbar = fig.colorbar(m, ax=axs[-1, :],location='bottom', cmap='Blues')

# Get the current position of the colorbar
pos = cbar.ax.get_position()

# Set the new position of the colorbar
cbar.ax.set_position([pos.x0-0.07, pos.y0-0.29, pos.width+0.2, pos.height])

axs[0,0].text(0, 1.07, "(a)", transform=axs[0,0].transAxes, fontsize=12, fontweight='bold', va='top')
axs[0,1].text(0.92, 1.07, "(b)", transform=axs[0,1].transAxes, fontsize=12, fontweight='bold', va='top')
axs[1,0].text(0, 1.07, "(c)", transform=axs[1,0].transAxes, fontsize=12, fontweight='bold', va='top')
axs[1,1].text(0.92, 1.07, "(d)", transform=axs[1,1].transAxes, fontsize=12, fontweight='bold', va='top')
axs[2,0].text(0, 1.07, "(e)", transform=axs[2,0].transAxes, fontsize=12, fontweight='bold', va='top')
axs[2,1].text(0.92, 1.07, "(f)", transform=axs[2,1].transAxes, fontsize=12, fontweight='bold', va='top')

axs[0,0].text(0.5, 1.16, "CMEMS", transform=axs[0,0].transAxes, fontsize=12, ha='center', va='top')
axs[0,1].text(0.5, 1.16, "ROMS", transform=axs[0,1].transAxes, fontsize=12, ha='center', va='top')

# Adjust spacing between subplots
plt.tight_layout()

fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/fig04", bbox_inches='tight', facecolor='white', transparent=False)


In [ ]:
# Figure 5

In [ ]:
escaped_file_CMEMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/escaped_beached_percentages/when_escaped_percentages_CMEMS.csv'
escaped_file_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/escaped_beached_percentages/when_escaped_percentages_ROMS.csv'
beached_file_CMEMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/escaped_beached_percentages/when_beached_percentages_CMEMS.csv'
beached_file_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/escaped_beached_percentages/when_beached_percentages_ROMS.csv'


In [ ]:
escaped_CMEMS = pd.read_csv(escaped_file_CMEMS, header=None)
escaped_ROMS = pd.read_csv(escaped_file_ROMS, header=None)
beached_CMEMS = pd.read_csv(beached_file_CMEMS, header=None)
beached_ROMS = pd.read_csv(beached_file_ROMS, header=None)

In [ ]:
# Subplot 1 data
week_beached_mon_values = [beached_CMEMS.iloc[0,0], beached_CMEMS.iloc[1,0], beached_CMEMS.iloc[2,0], beached_CMEMS.iloc[3,0], beached_CMEMS.iloc[4,0], beached_CMEMS.iloc[5,0]]  # c1 values
season_beached_mon_values = [beached_CMEMS.iloc[0,1], beached_CMEMS.iloc[1,1], beached_CMEMS.iloc[2,1], beached_CMEMS.iloc[3,1], beached_CMEMS.iloc[4,1], beached_CMEMS.iloc[5,1]]  # c1 values
later_beached_mon_values = [beached_CMEMS.iloc[0,2], beached_CMEMS.iloc[1,2], beached_CMEMS.iloc[2,2], beached_CMEMS.iloc[3,2], beached_CMEMS.iloc[4,2], beached_CMEMS.iloc[5,2]]  # c1 values

# Subplot 2 data
week_beached_post_values = [beached_CMEMS.iloc[0,3], beached_CMEMS.iloc[1,3], beached_CMEMS.iloc[2,3], beached_CMEMS.iloc[3,3], beached_CMEMS.iloc[4,3], beached_CMEMS.iloc[5,3]]  # c1 values
season_beached_post_values = [beached_CMEMS.iloc[0,4], beached_CMEMS.iloc[1,4], beached_CMEMS.iloc[2,4], beached_CMEMS.iloc[3,4], beached_CMEMS.iloc[4,4], beached_CMEMS.iloc[5,4]]  # c1 values
later_beached_post_values = [beached_CMEMS.iloc[0,5], beached_CMEMS.iloc[1,5], beached_CMEMS.iloc[2,5], beached_CMEMS.iloc[3,5], beached_CMEMS.iloc[4,5], beached_CMEMS.iloc[5,5]]  # c1 values

# Subplot 3 data
week_beached_pre_values = [beached_CMEMS.iloc[0,6], beached_CMEMS.iloc[1,6], beached_CMEMS.iloc[2,6], beached_CMEMS.iloc[3,6], beached_CMEMS.iloc[4,6], beached_CMEMS.iloc[5,6]]  # c1 values
season_beached_pre_values = [beached_CMEMS.iloc[0,7], beached_CMEMS.iloc[1,7], beached_CMEMS.iloc[2,7], beached_CMEMS.iloc[3,7], beached_CMEMS.iloc[4,7], beached_CMEMS.iloc[5,7]]  # c1 values
later_beached_pre_values = [beached_CMEMS.iloc[0,8], beached_CMEMS.iloc[1,8], beached_CMEMS.iloc[2,8], beached_CMEMS.iloc[3,8], beached_CMEMS.iloc[4,8], beached_CMEMS.iloc[5,8]]  # c1 values

# Subplot 1 data
week_beached_mon_valuesR = [beached_ROMS.iloc[0,0], beached_ROMS.iloc[1,0], beached_ROMS.iloc[2,0], beached_ROMS.iloc[3,0], beached_ROMS.iloc[4,0], beached_ROMS.iloc[5,0]]  # c1 values
season_beached_mon_valuesR = [beached_ROMS.iloc[0,1], beached_ROMS.iloc[1,1], beached_ROMS.iloc[2,1], beached_ROMS.iloc[3,1], beached_ROMS.iloc[4,1], beached_ROMS.iloc[5,1]]  # c1 values
later_beached_mon_valuesR = [beached_ROMS.iloc[0,2], beached_ROMS.iloc[1,2], beached_ROMS.iloc[2,2], beached_ROMS.iloc[3,2], beached_ROMS.iloc[4,2], beached_ROMS.iloc[5,2]]  # c1 values

# Subplot 2 data
week_beached_post_valuesR = [beached_ROMS.iloc[0,3], beached_ROMS.iloc[1,3], beached_ROMS.iloc[2,3], beached_ROMS.iloc[3,3], beached_ROMS.iloc[4,3], beached_ROMS.iloc[5,3]]  # c1 values
season_beached_post_valuesR = [beached_ROMS.iloc[0,4], beached_ROMS.iloc[1,4], beached_ROMS.iloc[2,4], beached_ROMS.iloc[3,4], beached_ROMS.iloc[4,4], beached_ROMS.iloc[5,4]]  # c1 values
later_beached_post_valuesR = [beached_ROMS.iloc[0,5], beached_ROMS.iloc[1,5], beached_ROMS.iloc[2,5], beached_ROMS.iloc[3,5], beached_ROMS.iloc[4,5], beached_ROMS.iloc[5,5]]  # c1 values

# Subplot 3 data
week_beached_pre_valuesR = [beached_ROMS.iloc[0,6], beached_ROMS.iloc[1,6], beached_ROMS.iloc[2,6], beached_ROMS.iloc[3,6], beached_ROMS.iloc[4,6], beached_ROMS.iloc[5,6]]  # c1 values
season_beached_pre_valuesR = [beached_ROMS.iloc[0,7], beached_ROMS.iloc[1,7], beached_ROMS.iloc[2,7], beached_ROMS.iloc[3,7], beached_ROMS.iloc[4,7], beached_ROMS.iloc[5,7]]  # c1 values
later_beached_pre_valuesR = [beached_ROMS.iloc[0,8], beached_ROMS.iloc[1,8], beached_ROMS.iloc[2,8], beached_ROMS.iloc[3,8], beached_ROMS.iloc[4,8], beached_ROMS.iloc[5,8]]  # c1 values


In [ ]:
# Subplot 1 data
week_escaped_mon_values = [escaped_CMEMS.iloc[0,0], escaped_CMEMS.iloc[1,0], escaped_CMEMS.iloc[2,0], escaped_CMEMS.iloc[3,0], escaped_CMEMS.iloc[4,0], escaped_CMEMS.iloc[5,0]]  # c1 values
season_escaped_mon_values = [escaped_CMEMS.iloc[0,1], escaped_CMEMS.iloc[1,1], escaped_CMEMS.iloc[2,1], escaped_CMEMS.iloc[3,1], escaped_CMEMS.iloc[4,1], escaped_CMEMS.iloc[5,1]]  # c1 values
later_escaped_mon_values = [escaped_CMEMS.iloc[0,2], escaped_CMEMS.iloc[1,2], escaped_CMEMS.iloc[2,2], escaped_CMEMS.iloc[3,2], escaped_CMEMS.iloc[4,2], escaped_CMEMS.iloc[5,2]]  # c1 values

# Subplot 2 data
week_escaped_post_values = [escaped_CMEMS.iloc[0,3], escaped_CMEMS.iloc[1,3], escaped_CMEMS.iloc[2,3], escaped_CMEMS.iloc[3,3], escaped_CMEMS.iloc[4,3], escaped_CMEMS.iloc[5,3]]  # c1 values
season_escaped_post_values = [escaped_CMEMS.iloc[0,4], escaped_CMEMS.iloc[1,4], escaped_CMEMS.iloc[2,4], escaped_CMEMS.iloc[3,4], escaped_CMEMS.iloc[4,4], escaped_CMEMS.iloc[5,4]]  # c1 values
later_escaped_post_values = [escaped_CMEMS.iloc[0,5], escaped_CMEMS.iloc[1,5], escaped_CMEMS.iloc[2,5], escaped_CMEMS.iloc[3,5], escaped_CMEMS.iloc[4,5], escaped_CMEMS.iloc[5,5]]  # c1 values

# Subplot 3 data
week_escaped_pre_values = [escaped_CMEMS.iloc[0,6], escaped_CMEMS.iloc[1,6], escaped_CMEMS.iloc[2,6], escaped_CMEMS.iloc[3,6], escaped_CMEMS.iloc[4,6], escaped_CMEMS.iloc[5,6]]  # c1 values
season_escaped_pre_values = [escaped_CMEMS.iloc[0,7], escaped_CMEMS.iloc[1,7], escaped_CMEMS.iloc[2,7], escaped_CMEMS.iloc[3,7], escaped_CMEMS.iloc[4,7], escaped_CMEMS.iloc[5,7]]  # c1 values
later_escaped_pre_values = [escaped_CMEMS.iloc[0,8], escaped_CMEMS.iloc[1,8], escaped_CMEMS.iloc[2,8], escaped_CMEMS.iloc[3,8], escaped_CMEMS.iloc[4,8], escaped_CMEMS.iloc[5,8]]  # c1 values

# Subplot 1 data
week_escaped_mon_valuesR = [escaped_ROMS.iloc[0,0], escaped_ROMS.iloc[1,0], escaped_ROMS.iloc[2,0], escaped_ROMS.iloc[3,0], escaped_ROMS.iloc[4,0], escaped_ROMS.iloc[5,0]]  # c1 values
season_escaped_mon_valuesR = [escaped_ROMS.iloc[0,1], escaped_ROMS.iloc[1,1], escaped_ROMS.iloc[2,1], escaped_ROMS.iloc[3,1], escaped_ROMS.iloc[4,1], escaped_ROMS.iloc[5,1]]  # c1 values
later_escaped_mon_valuesR = [escaped_ROMS.iloc[0,2], escaped_ROMS.iloc[1,2], escaped_ROMS.iloc[2,2], escaped_ROMS.iloc[3,2], escaped_ROMS.iloc[4,2], escaped_ROMS.iloc[5,2]]  # c1 values

# Subplot 2 data
week_escaped_post_valuesR = [escaped_ROMS.iloc[0,3], escaped_ROMS.iloc[1,3], escaped_ROMS.iloc[2,3], escaped_ROMS.iloc[3,3], escaped_ROMS.iloc[4,3], escaped_ROMS.iloc[5,3]]  # c1 values
season_escaped_post_valuesR = [escaped_ROMS.iloc[0,4], escaped_ROMS.iloc[1,4], escaped_ROMS.iloc[2,4], escaped_ROMS.iloc[3,4], escaped_ROMS.iloc[4,4], escaped_ROMS.iloc[5,4]]  # c1 values
later_escaped_post_valuesR = [escaped_ROMS.iloc[0,5], escaped_ROMS.iloc[1,5], escaped_ROMS.iloc[2,5], escaped_ROMS.iloc[3,5], escaped_ROMS.iloc[4,5], escaped_ROMS.iloc[5,5]]  # c1 values

# Subplot 3 data
week_escaped_pre_valuesR = [escaped_ROMS.iloc[0,6], escaped_ROMS.iloc[1,6], escaped_ROMS.iloc[2,6], escaped_ROMS.iloc[3,6], escaped_ROMS.iloc[4,6], escaped_ROMS.iloc[5,6]]  # c1 values
season_escaped_pre_valuesR = [escaped_ROMS.iloc[0,7], escaped_ROMS.iloc[1,7], escaped_ROMS.iloc[2,7], escaped_ROMS.iloc[3,7], escaped_ROMS.iloc[4,7], escaped_ROMS.iloc[5,7]]  # c1 values
later_escaped_pre_valuesR = [escaped_ROMS.iloc[0,8], escaped_ROMS.iloc[1,8], escaped_ROMS.iloc[2,8], escaped_ROMS.iloc[3,8], escaped_ROMS.iloc[4,8], escaped_ROMS.iloc[5,8]]  # c1 values

In [ ]:
categories = ['Beached within a week', 'Beached during rest of season', 'Beached later']
bar_width = 0.05
space = 1.0  # Space between categories
plt.rcParams['hatch.linewidth'] = 4.0

# Create figure and subplots
fig, axes = plt.subplots(3, 1, figsize=(10, 13), sharey=True)

x_positions = np.arange(len(categories)) * 18 * bar_width + space
labels = ['Sri Lanka', 'India', 'Bangladesh', 'Myanmar', 'Thailand', 'Indonesia']

# Subplot 1
axes[0].bar(x_positions[0] - 5.5 * bar_width, week_beached_mon_values[0], width=bar_width, color='lightpink', edgecolor='white', label=labels[0])
axes[0].bar(x_positions[0] - 3.5 * bar_width, week_beached_mon_values[1], width=bar_width, color='gold', edgecolor='white', label=labels[1])
axes[0].bar(x_positions[0] - 1.5 * bar_width, week_beached_mon_values[2], width=bar_width, color='crimson', edgecolor='white', label=labels[2])
axes[0].bar(x_positions[0] + 0.5 * bar_width, week_beached_mon_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white', label=labels[3])
axes[0].bar(x_positions[0] + 2.5 * bar_width, week_beached_mon_values[4], width=bar_width, color='k', edgecolor='white', label=labels[4])
axes[0].bar(x_positions[0] + 4.5 * bar_width, week_beached_mon_values[5], width=bar_width, color='cyan', edgecolor='white', label=labels[5])
axes[0].bar(x_positions[1] - 5.5 * bar_width, season_beached_mon_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[0].bar(x_positions[1] - 3.5 * bar_width, season_beached_mon_values[1], width=bar_width, color='gold', edgecolor='white')
axes[0].bar(x_positions[1] - 1.5 * bar_width, season_beached_mon_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[0].bar(x_positions[1] + 0.5 * bar_width, season_beached_mon_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[0].bar(x_positions[1] + 2.5 * bar_width, season_beached_mon_values[4], width=bar_width, color='k', edgecolor='white')
axes[0].bar(x_positions[1] + 4.5 * bar_width, season_beached_mon_values[5], width=bar_width, color='cyan', edgecolor='white')
axes[0].bar(x_positions[2] - 5.5 * bar_width, later_beached_mon_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[0].bar(x_positions[2] - 3.5 * bar_width, later_beached_mon_values[1], width=bar_width, color='gold', edgecolor='white')
axes[0].bar(x_positions[2] - 1.5 * bar_width, later_beached_mon_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[0].bar(x_positions[2] + 0.5 * bar_width, later_beached_mon_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[0].bar(x_positions[2] + 2.5 * bar_width, later_beached_mon_values[4], width=bar_width, color='k', edgecolor='white')
axes[0].bar(x_positions[2] + 4.5 * bar_width, later_beached_mon_values[5], width=bar_width, color='cyan', edgecolor='white')

axes[0].bar(x_positions[0] - 4.5 * bar_width, week_beached_mon_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[0].bar(x_positions[0] - 2.5 * bar_width, week_beached_mon_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[0].bar(x_positions[0] - 0.5 * bar_width, week_beached_mon_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[0].bar(x_positions[0] + 1.5 * bar_width, week_beached_mon_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[0].bar(x_positions[0] + 3.5 * bar_width, week_beached_mon_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[0].bar(x_positions[0] + 5.5 * bar_width, week_beached_mon_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')
axes[0].bar(x_positions[1] - 4.5 * bar_width, season_beached_mon_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[0].bar(x_positions[1] - 2.5 * bar_width, season_beached_mon_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[0].bar(x_positions[1] - 0.5 * bar_width, season_beached_mon_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[0].bar(x_positions[1] + 1.5 * bar_width, season_beached_mon_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[0].bar(x_positions[1] + 3.5 * bar_width, season_beached_mon_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[0].bar(x_positions[1] + 5.5 * bar_width, season_beached_mon_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')
axes[0].bar(x_positions[2] - 4.5 * bar_width, later_beached_mon_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[0].bar(x_positions[2] - 2.5 * bar_width, later_beached_mon_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[0].bar(x_positions[2] - 0.5 * bar_width, later_beached_mon_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[0].bar(x_positions[2] + 1.5 * bar_width, later_beached_mon_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[0].bar(x_positions[2] + 3.5 * bar_width, later_beached_mon_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[0].bar(x_positions[2] + 5.5 * bar_width, later_beached_mon_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')

axes[0].set_xticks(x_positions)
axes[0].set_xticklabels(categories, fontsize=11)
axes[0].set_title("Monsoon (Jun - Sept)", fontsize=12)
axes[0].yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
for label in axes[0].get_yticklabels():
    label.set_fontsize(11)
axes[0].legend(fontsize=11)

# Subplot 2
axes[1].bar(x_positions[0] - 5.5 * bar_width, week_beached_post_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[1].bar(x_positions[0] - 3.5 * bar_width, week_beached_post_values[1], width=bar_width, color='gold', edgecolor='white')
axes[1].bar(x_positions[0] - 1.5 * bar_width, week_beached_post_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[1].bar(x_positions[0] + 0.5 * bar_width, week_beached_post_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[1].bar(x_positions[0] + 2.5 * bar_width, week_beached_post_values[4], width=bar_width, color='k', edgecolor='white')
axes[1].bar(x_positions[0] + 4.5 * bar_width, week_beached_post_values[5], width=bar_width, color='cyan', edgecolor='white')
axes[1].bar(x_positions[1] - 5.5 * bar_width, season_beached_post_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[1].bar(x_positions[1] - 3.5 * bar_width, season_beached_post_values[1], width=bar_width, color='gold', edgecolor='white')
axes[1].bar(x_positions[1] - 1.5 * bar_width, season_beached_post_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[1].bar(x_positions[1] + 0.5 * bar_width, season_beached_post_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[1].bar(x_positions[1] + 2.5 * bar_width, season_beached_post_values[4], width=bar_width, color='k', edgecolor='white')
axes[1].bar(x_positions[1] + 4.5 * bar_width, season_beached_post_values[5], width=bar_width, color='cyan', edgecolor='white')
axes[1].bar(x_positions[2] - 5.5 * bar_width, later_beached_post_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[1].bar(x_positions[2] - 3.5 * bar_width, later_beached_post_values[1], width=bar_width, color='gold', edgecolor='white')
axes[1].bar(x_positions[2] - 1.5 * bar_width, later_beached_post_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[1].bar(x_positions[2] + 0.5 * bar_width, later_beached_post_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[1].bar(x_positions[2] + 2.5 * bar_width, later_beached_post_values[4], width=bar_width, color='k', edgecolor='white')
axes[1].bar(x_positions[2] + 4.5 * bar_width, later_beached_post_values[5], width=bar_width, color='cyan', edgecolor='white')

axes[1].bar(x_positions[0] - 4.5 * bar_width, week_beached_post_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[1].bar(x_positions[0] - 2.5 * bar_width, week_beached_post_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[1].bar(x_positions[0] - 0.5 * bar_width, week_beached_post_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[1].bar(x_positions[0] + 1.5 * bar_width, week_beached_post_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[1].bar(x_positions[0] + 3.5 * bar_width, week_beached_post_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[1].bar(x_positions[0] + 5.5 * bar_width, week_beached_post_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')
axes[1].bar(x_positions[1] - 4.5 * bar_width, season_beached_post_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[1].bar(x_positions[1] - 2.5 * bar_width, season_beached_post_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[1].bar(x_positions[1] - 0.5 * bar_width, season_beached_post_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[1].bar(x_positions[1] + 1.5 * bar_width, season_beached_post_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[1].bar(x_positions[1] + 3.5 * bar_width, season_beached_post_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[1].bar(x_positions[1] + 5.5 * bar_width, season_beached_post_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')
axes[1].bar(x_positions[2] - 4.5 * bar_width, later_beached_post_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[1].bar(x_positions[2] - 2.5 * bar_width, later_beached_post_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[1].bar(x_positions[2] - 0.5 * bar_width, later_beached_post_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[1].bar(x_positions[2] + 1.5 * bar_width, later_beached_post_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[1].bar(x_positions[2] + 3.5 * bar_width, later_beached_post_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[1].bar(x_positions[2] + 5.5 * bar_width, later_beached_post_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')

axes[1].set_xticks(x_positions)
axes[1].set_xticklabels(categories, fontsize=11)
axes[1].set_title("Post-monsoon (Oct - Jan)", fontsize=12)
axes[1].yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
for label in axes[1].get_yticklabels():
    label.set_fontsize(11)

# Subplot 3
axes[2].bar(x_positions[0] - 5.5 * bar_width, week_beached_pre_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[2].bar(x_positions[0] - 3.5 * bar_width, week_beached_pre_values[1], width=bar_width, color='gold', edgecolor='white')
axes[2].bar(x_positions[0] - 1.5 * bar_width, week_beached_pre_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[2].bar(x_positions[0] + 0.5 * bar_width, week_beached_pre_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[2].bar(x_positions[0] + 2.5 * bar_width, week_beached_pre_values[4], width=bar_width, color='k', edgecolor='white')
axes[2].bar(x_positions[0] + 4.5 * bar_width, week_beached_pre_values[5], width=bar_width, color='cyan', edgecolor='white')
axes[2].bar(x_positions[1] - 5.5 * bar_width, season_beached_pre_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[2].bar(x_positions[1] - 3.5 * bar_width, season_beached_pre_values[1], width=bar_width, color='gold', edgecolor='white')
axes[2].bar(x_positions[1] - 1.5 * bar_width, season_beached_pre_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[2].bar(x_positions[1] + 0.5 * bar_width, season_beached_pre_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[2].bar(x_positions[1] + 2.5 * bar_width, season_beached_pre_values[4], width=bar_width, color='k', edgecolor='white')
axes[2].bar(x_positions[1] + 4.5 * bar_width, season_beached_pre_values[5], width=bar_width, color='cyan', edgecolor='white')
axes[2].bar(x_positions[2] - 5.5 * bar_width, later_beached_pre_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[2].bar(x_positions[2] - 3.5 * bar_width, later_beached_pre_values[1], width=bar_width, color='gold', edgecolor='white')
axes[2].bar(x_positions[2] - 1.5 * bar_width, later_beached_pre_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[2].bar(x_positions[2] + 0.5 * bar_width, later_beached_pre_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[2].bar(x_positions[2] + 2.5 * bar_width, later_beached_pre_values[4], width=bar_width, color='k', edgecolor='white')
axes[2].bar(x_positions[2] + 4.5 * bar_width, later_beached_pre_values[5], width=bar_width, color='cyan', edgecolor='white')

axes[2].bar(x_positions[0] - 4.5 * bar_width, week_beached_pre_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[2].bar(x_positions[0] - 2.5 * bar_width, week_beached_pre_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[2].bar(x_positions[0] - 0.5 * bar_width, week_beached_pre_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[2].bar(x_positions[0] + 1.5 * bar_width, week_beached_pre_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[2].bar(x_positions[0] + 3.5 * bar_width, week_beached_pre_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[2].bar(x_positions[0] + 5.5 * bar_width, week_beached_pre_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')
axes[2].bar(x_positions[1] - 4.5 * bar_width, season_beached_pre_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[2].bar(x_positions[1] - 2.5 * bar_width, season_beached_pre_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[2].bar(x_positions[1] - 0.5 * bar_width, season_beached_pre_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[2].bar(x_positions[1] + 1.5 * bar_width, season_beached_pre_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[2].bar(x_positions[1] + 3.5 * bar_width, season_beached_pre_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[2].bar(x_positions[1] + 5.5 * bar_width, season_beached_pre_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')
axes[2].bar(x_positions[2] - 4.5 * bar_width, later_beached_pre_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[2].bar(x_positions[2] - 2.5 * bar_width, later_beached_pre_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[2].bar(x_positions[2] - 0.5 * bar_width, later_beached_pre_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[2].bar(x_positions[2] + 1.5 * bar_width, later_beached_pre_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[2].bar(x_positions[2] + 3.5 * bar_width, later_beached_pre_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[2].bar(x_positions[2] + 5.5 * bar_width, later_beached_pre_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')

axes[2].set_xticks(x_positions)
axes[2].set_xticklabels(categories,fontsize=11)
axes[2].set_title("Pre-monsoon (Feb - May)", fontsize=12)
axes[2].yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
for label in axes[2].get_yticklabels():
    label.set_fontsize(11)

axes[0].text(0, 1.07, "(a)", transform=axes[0].transAxes, fontsize=12, fontweight='bold', va='top')
axes[1].text(0, 1.07, "(b)", transform=axes[1].transAxes, fontsize=12, fontweight='bold', va='top')
axes[2].text(0, 1.07, "(c)", transform=axes[2].transAxes, fontsize=12, fontweight='bold', va='top')

fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/fig05", bbox_inches='tight')

In [ ]:
# Figure 6

In [ ]:
categories_escaped = ['Exited domain within a week', 'Exited domain during rest of season', 'Exited domain later']
bar_width = 0.05
space = 1.0  # Space between categories
plt.rcParams['hatch.linewidth'] = 4.0

# Create figure and subplots
fig, axes = plt.subplots(3, 1, figsize=(10, 13), sharey=True)

x_positions = np.arange(len(categories_escaped)) * 18 * bar_width + space
labels = ['Sri Lanka', 'India', 'Bangladesh', 'Myanmar', 'Thailand', 'Indonesia']

# Subplot 1
axes[0].bar(x_positions[0] - 5.5 * bar_width, week_escaped_mon_values[0], width=bar_width, color='lightpink', edgecolor='white', label=labels[0])
axes[0].bar(x_positions[0] - 3.5 * bar_width, week_escaped_mon_values[1], width=bar_width, color='gold', edgecolor='white', label=labels[1])
axes[0].bar(x_positions[0] - 1.5 * bar_width, week_escaped_mon_values[2], width=bar_width, color='crimson', edgecolor='white', label=labels[2])
axes[0].bar(x_positions[0] + 0.5 * bar_width, week_escaped_mon_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white', label=labels[3])
axes[0].bar(x_positions[0] + 2.5 * bar_width, week_escaped_mon_values[4], width=bar_width, color='k', edgecolor='white', label=labels[4])
axes[0].bar(x_positions[0] + 4.5 * bar_width, week_escaped_mon_values[5], width=bar_width, color='cyan', edgecolor='white', label=labels[5])
axes[0].bar(x_positions[1] - 5.5 * bar_width, season_escaped_mon_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[0].bar(x_positions[1] - 3.5 * bar_width, season_escaped_mon_values[1], width=bar_width, color='gold', edgecolor='white')
axes[0].bar(x_positions[1] - 1.5 * bar_width, season_escaped_mon_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[0].bar(x_positions[1] + 0.5 * bar_width, season_escaped_mon_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[0].bar(x_positions[1] + 2.5 * bar_width, season_escaped_mon_values[4], width=bar_width, color='k', edgecolor='white')
axes[0].bar(x_positions[1] + 4.5 * bar_width, season_escaped_mon_values[5], width=bar_width, color='cyan', edgecolor='white')
axes[0].bar(x_positions[2] - 5.5 * bar_width, later_escaped_mon_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[0].bar(x_positions[2] - 3.5 * bar_width, later_escaped_mon_values[1], width=bar_width, color='gold', edgecolor='white')
axes[0].bar(x_positions[2] - 1.5 * bar_width, later_escaped_mon_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[0].bar(x_positions[2] + 0.5 * bar_width, later_escaped_mon_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[0].bar(x_positions[2] + 2.5 * bar_width, later_escaped_mon_values[4], width=bar_width, color='k', edgecolor='white')
axes[0].bar(x_positions[2] + 4.5 * bar_width, later_escaped_mon_values[5], width=bar_width, color='cyan', edgecolor='white')

axes[0].bar(x_positions[0] - 4.5 * bar_width, week_escaped_mon_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[0].bar(x_positions[0] - 2.5 * bar_width, week_escaped_mon_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[0].bar(x_positions[0] - 0.5 * bar_width, week_escaped_mon_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[0].bar(x_positions[0] + 1.5 * bar_width, week_escaped_mon_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[0].bar(x_positions[0] + 3.5 * bar_width, week_escaped_mon_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[0].bar(x_positions[0] + 5.5 * bar_width, week_escaped_mon_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')
axes[0].bar(x_positions[1] - 4.5 * bar_width, season_escaped_mon_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[0].bar(x_positions[1] - 2.5 * bar_width, season_escaped_mon_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[0].bar(x_positions[1] - 0.5 * bar_width, season_escaped_mon_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[0].bar(x_positions[1] + 1.5 * bar_width, season_escaped_mon_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[0].bar(x_positions[1] + 3.5 * bar_width, season_escaped_mon_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[0].bar(x_positions[1] + 5.5 * bar_width, season_escaped_mon_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')
axes[0].bar(x_positions[2] - 4.5 * bar_width, later_escaped_mon_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[0].bar(x_positions[2] - 2.5 * bar_width, later_escaped_mon_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[0].bar(x_positions[2] - 0.5 * bar_width, later_escaped_mon_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[0].bar(x_positions[2] + 1.5 * bar_width, later_escaped_mon_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[0].bar(x_positions[2] + 3.5 * bar_width, later_escaped_mon_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[0].bar(x_positions[2] + 5.5 * bar_width, later_escaped_mon_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')

axes[0].set_xticks(x_positions)
axes[0].set_xticklabels(categories_escaped, fontsize=11)
axes[0].set_title("Monsoon (Jun - Sept)", fontsize=12)
axes[0].yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
for label in axes[0].get_yticklabels():
    label.set_fontsize(11)
axes[0].legend(fontsize=11)

# Subplot 2
axes[1].bar(x_positions[0] - 5.5 * bar_width, week_escaped_post_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[1].bar(x_positions[0] - 3.5 * bar_width, week_escaped_post_values[1], width=bar_width, color='gold', edgecolor='white')
axes[1].bar(x_positions[0] - 1.5 * bar_width, week_escaped_post_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[1].bar(x_positions[0] + 0.5 * bar_width, week_escaped_post_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[1].bar(x_positions[0] + 2.5 * bar_width, week_escaped_post_values[4], width=bar_width, color='k', edgecolor='white')
axes[1].bar(x_positions[0] + 4.5 * bar_width, week_escaped_post_values[5], width=bar_width, color='cyan', edgecolor='white')
axes[1].bar(x_positions[1] - 5.5 * bar_width, season_escaped_post_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[1].bar(x_positions[1] - 3.5 * bar_width, season_escaped_post_values[1], width=bar_width, color='gold', edgecolor='white')
axes[1].bar(x_positions[1] - 1.5 * bar_width, season_escaped_post_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[1].bar(x_positions[1] + 0.5 * bar_width, season_escaped_post_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[1].bar(x_positions[1] + 2.5 * bar_width, season_escaped_post_values[4], width=bar_width, color='k', edgecolor='white')
axes[1].bar(x_positions[1] + 4.5 * bar_width, season_escaped_post_values[5], width=bar_width, color='cyan', edgecolor='white')
axes[1].bar(x_positions[2] - 5.5 * bar_width, later_escaped_post_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[1].bar(x_positions[2] - 3.5 * bar_width, later_escaped_post_values[1], width=bar_width, color='gold', edgecolor='white')
axes[1].bar(x_positions[2] - 1.5 * bar_width, later_escaped_post_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[1].bar(x_positions[2] + 0.5 * bar_width, later_escaped_post_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[1].bar(x_positions[2] + 2.5 * bar_width, later_escaped_post_values[4], width=bar_width, color='k', edgecolor='white')
axes[1].bar(x_positions[2] + 4.5 * bar_width, later_escaped_post_values[5], width=bar_width, color='cyan', edgecolor='white')

axes[1].bar(x_positions[0] - 4.5 * bar_width, week_escaped_post_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[1].bar(x_positions[0] - 2.5 * bar_width, week_escaped_post_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[1].bar(x_positions[0] - 0.5 * bar_width, week_escaped_post_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[1].bar(x_positions[0] + 1.5 * bar_width, week_escaped_post_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[1].bar(x_positions[0] + 3.5 * bar_width, week_escaped_post_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[1].bar(x_positions[0] + 5.5 * bar_width, week_escaped_post_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')
axes[1].bar(x_positions[1] - 4.5 * bar_width, season_escaped_post_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[1].bar(x_positions[1] - 2.5 * bar_width, season_escaped_post_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[1].bar(x_positions[1] - 0.5 * bar_width, season_escaped_post_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[1].bar(x_positions[1] + 1.5 * bar_width, season_escaped_post_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[1].bar(x_positions[1] + 3.5 * bar_width, season_escaped_post_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[1].bar(x_positions[1] + 5.5 * bar_width, season_escaped_post_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')
axes[1].bar(x_positions[2] - 4.5 * bar_width, later_escaped_post_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[1].bar(x_positions[2] - 2.5 * bar_width, later_escaped_post_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[1].bar(x_positions[2] - 0.5 * bar_width, later_escaped_post_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[1].bar(x_positions[2] + 1.5 * bar_width, later_escaped_post_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[1].bar(x_positions[2] + 3.5 * bar_width, later_escaped_post_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[1].bar(x_positions[2] + 5.5 * bar_width, later_escaped_post_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')

axes[1].set_xticks(x_positions)
axes[1].set_xticklabels(categories_escaped, fontsize=11)
axes[1].set_title("Post-monsoon (Oct - Jan)", fontsize=12)
axes[1].yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
for label in axes[1].get_yticklabels():
    label.set_fontsize(11)

# Subplot 3
axes[2].bar(x_positions[0] - 5.5 * bar_width, week_escaped_pre_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[2].bar(x_positions[0] - 3.5 * bar_width, week_escaped_pre_values[1], width=bar_width, color='gold', edgecolor='white')
axes[2].bar(x_positions[0] - 1.5 * bar_width, week_escaped_pre_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[2].bar(x_positions[0] + 0.5 * bar_width, week_escaped_pre_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[2].bar(x_positions[0] + 2.5 * bar_width, week_escaped_pre_values[4], width=bar_width, color='k', edgecolor='white')
axes[2].bar(x_positions[0] + 4.5 * bar_width, week_escaped_pre_values[5], width=bar_width, color='cyan', edgecolor='white')
axes[2].bar(x_positions[1] - 5.5 * bar_width, season_escaped_pre_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[2].bar(x_positions[1] - 3.5 * bar_width, season_escaped_pre_values[1], width=bar_width, color='gold', edgecolor='white')
axes[2].bar(x_positions[1] - 1.5 * bar_width, season_escaped_pre_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[2].bar(x_positions[1] + 0.5 * bar_width, season_escaped_pre_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[2].bar(x_positions[1] + 2.5 * bar_width, season_escaped_pre_values[4], width=bar_width, color='k', edgecolor='white')
axes[2].bar(x_positions[1] + 4.5 * bar_width, season_escaped_pre_values[5], width=bar_width, color='cyan', edgecolor='white')
axes[2].bar(x_positions[2] - 5.5 * bar_width, later_escaped_pre_values[0], width=bar_width, color='lightpink', edgecolor='white')
axes[2].bar(x_positions[2] - 3.5 * bar_width, later_escaped_pre_values[1], width=bar_width, color='gold', edgecolor='white')
axes[2].bar(x_positions[2] - 1.5 * bar_width, later_escaped_pre_values[2], width=bar_width, color='crimson', edgecolor='white')
axes[2].bar(x_positions[2] + 0.5 * bar_width, later_escaped_pre_values[3], width=bar_width, color='mediumaquamarine', edgecolor='white')
axes[2].bar(x_positions[2] + 2.5 * bar_width, later_escaped_pre_values[4], width=bar_width, color='k', edgecolor='white')
axes[2].bar(x_positions[2] + 4.5 * bar_width, later_escaped_pre_values[5], width=bar_width, color='cyan', edgecolor='white')

axes[2].bar(x_positions[0] - 4.5 * bar_width, week_escaped_pre_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[2].bar(x_positions[0] - 2.5 * bar_width, week_escaped_pre_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[2].bar(x_positions[0] - 0.5 * bar_width, week_escaped_pre_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[2].bar(x_positions[0] + 1.5 * bar_width, week_escaped_pre_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[2].bar(x_positions[0] + 3.5 * bar_width, week_escaped_pre_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[2].bar(x_positions[0] + 5.5 * bar_width, week_escaped_pre_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')
axes[2].bar(x_positions[1] - 4.5 * bar_width, season_escaped_pre_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[2].bar(x_positions[1] - 2.5 * bar_width, season_escaped_pre_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[2].bar(x_positions[1] - 0.5 * bar_width, season_escaped_pre_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[2].bar(x_positions[1] + 1.5 * bar_width, season_escaped_pre_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[2].bar(x_positions[1] + 3.5 * bar_width, season_escaped_pre_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[2].bar(x_positions[1] + 5.5 * bar_width, season_escaped_pre_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')
axes[2].bar(x_positions[2] - 4.5 * bar_width, later_escaped_pre_valuesR[0], width=bar_width, color='lightpink', edgecolor='white', hatch='//')
axes[2].bar(x_positions[2] - 2.5 * bar_width, later_escaped_pre_valuesR[1], width=bar_width, color='gold', edgecolor='white', hatch='//')
axes[2].bar(x_positions[2] - 0.5 * bar_width, later_escaped_pre_valuesR[2], width=bar_width, color='crimson', edgecolor='white', hatch='//')
axes[2].bar(x_positions[2] + 1.5 * bar_width, later_escaped_pre_valuesR[3], width=bar_width, color='mediumaquamarine', edgecolor='white', hatch='//')
axes[2].bar(x_positions[2] + 3.5 * bar_width, later_escaped_pre_valuesR[4], width=bar_width, color='k', edgecolor='white', hatch='//')
axes[2].bar(x_positions[2] + 5.5 * bar_width, later_escaped_pre_valuesR[5], width=bar_width, color='cyan', edgecolor='white', hatch='//')

axes[2].set_xticks(x_positions)
axes[2].set_xticklabels(categories_escaped, fontsize=11)
axes[2].set_title("Pre-monsoon (Feb - May)", fontsize=12)
axes[2].yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
for label in axes[2].get_yticklabels():
    label.set_fontsize(11)

axes[0].text(0, 1.07, "(a)", transform=axes[0].transAxes, fontsize=12, fontweight='bold', va='top')
axes[1].text(0, 1.07, "(b)", transform=axes[1].transAxes, fontsize=12, fontweight='bold', va='top')
axes[2].text(0, 1.07, "(c)", transform=axes[2].transAxes, fontsize=12, fontweight='bold', va='top')

fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/fig06", bbox_inches='tight')


In [ ]:
# Appendices

In [ ]:
# Figure A1

In [ ]:
particle_input_file_SL_Cop = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_SL_uniform_Cop_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_India_Cop = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_India_uniform_Cop_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_Bang_Cop = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Bang_uniform_Cop_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_Myan_Cop = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Myan_uniform_Cop_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_Thai_Cop = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Thai_uniform_Cop_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_Indo_Cop = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Indonesia_uniform_Cop_daily_Jun2018-Sept2019_monsoon.nc'

# import data
openfile_p9_Cop = Dataset(particle_input_file_SL_Cop)
lon_p_SL_Cop = openfile_p9_Cop.variables['lon']
lat_p_SL_Cop = openfile_p9_Cop.variables['lat']
time_p_SL_Cop = openfile_p9_Cop.variables['time']
openfile_p10_Cop = Dataset(particle_input_file_India_Cop)
lon_p_India_Cop = openfile_p10_Cop.variables['lon']
lat_p_India_Cop = openfile_p10_Cop.variables['lat']
openfile_p11_Cop = Dataset(particle_input_file_Bang_Cop)
lon_p_Bang_Cop = openfile_p11_Cop.variables['lon']
lat_p_Bang_Cop = openfile_p11_Cop.variables['lat']
openfile_p12_Cop = Dataset(particle_input_file_Myan_Cop)
lon_p_Myan_Cop = openfile_p12_Cop.variables['lon']
lat_p_Myan_Cop = openfile_p12_Cop.variables['lat']
openfile_p13_Cop = Dataset(particle_input_file_Thai_Cop)
lon_p_Thai_Cop = openfile_p13_Cop.variables['lon']
lat_p_Thai_Cop = openfile_p13_Cop.variables['lat']
openfile_p14_Cop = Dataset(particle_input_file_Indo_Cop)
lon_p_Indo_Cop = openfile_p14_Cop.variables['lon']
lat_p_Indo_Cop = openfile_p14_Cop.variables['lat']


In [ ]:
particle_input_file_SL_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_SL_uniform_ROMS_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_India_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_India_uniform_ROMS_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_Bang_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Bang_uniform_ROMS_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_Myan_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Myan_uniform_ROMS_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_Thai_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Thai_uniform_ROMS_daily_Jun2018-Sept2019_monsoon.nc'
particle_input_file_Indo_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Indonesia_uniform_ROMS_daily_Jun2018-Sept2019_monsoon.nc'

# import data
openfile_p9_ROMS = Dataset(particle_input_file_SL_ROMS)
lon_p_SL_ROMS = openfile_p9_ROMS.variables['lon']
lat_p_SL_ROMS = openfile_p9_ROMS.variables['lat']
time_p_SL_ROMS = openfile_p9_ROMS.variables['time']
openfile_p10_ROMS = Dataset(particle_input_file_India_ROMS)
lon_p_India_ROMS = openfile_p10_ROMS.variables['lon']
lat_p_India_ROMS = openfile_p10_ROMS.variables['lat']
openfile_p11_ROMS = Dataset(particle_input_file_Bang_ROMS)
lon_p_Bang_ROMS = openfile_p11_ROMS.variables['lon']
lat_p_Bang_ROMS = openfile_p11_ROMS.variables['lat']
openfile_p12_ROMS = Dataset(particle_input_file_Myan_ROMS)
lon_p_Myan_ROMS = openfile_p12_ROMS.variables['lon']
lat_p_Myan_ROMS = openfile_p12_ROMS.variables['lat']
openfile_p13_ROMS = Dataset(particle_input_file_Thai_ROMS)
lon_p_Thai_ROMS = openfile_p13_ROMS.variables['lon']
lat_p_Thai_ROMS = openfile_p13_ROMS.variables['lat']
openfile_p14_ROMS = Dataset(particle_input_file_Indo_ROMS)
lon_p_Indo_ROMS = openfile_p14_ROMS.variables['lon']
lat_p_Indo_ROMS = openfile_p14_ROMS.variables['lat']


In [ ]:
beached_p_SL_Cop = openfile_p9_Cop.variables['beached']
beached_p_India_Cop = openfile_p10_Cop.variables['beached']
beached_p_Bang_Cop = openfile_p11_Cop.variables['beached']
beached_p_Myan_Cop = openfile_p12_Cop.variables['beached']
beached_p_Thai_Cop = openfile_p13_Cop.variables['beached']
beached_p_Indo_Cop = openfile_p14_Cop.variables['beached']

In [ ]:
beached_p_SL_ROMS = openfile_p9_ROMS.variables['beached']
beached_p_India_ROMS = openfile_p10_ROMS.variables['beached']
beached_p_Bang_ROMS = openfile_p11_ROMS.variables['beached']
beached_p_Myan_ROMS = openfile_p12_ROMS.variables['beached']
beached_p_Thai_ROMS = openfile_p13_ROMS.variables['beached']
beached_p_Indo_ROMS = openfile_p14_ROMS.variables['beached']

In [ ]:
lon_p_beached_SL_Cop = np.zeros(len(lon_p_SL_Cop))
lat_p_beached_SL_Cop = np.zeros(len(lat_p_SL_Cop))
for ii in range(len(lon_p_SL_Cop)):
    lon_temp = lon_p_SL_Cop[ii,:]
    lat_temp = lat_p_SL_Cop[ii,:]
    beached_temp = beached_p_SL_Cop[ii,:]
    for jj in range(len(lon_temp)):
        if beached_temp[jj]==1:
            if not lon_temp[jj].data==0:
               lon_p_beached_SL_Cop[ii] = lon_temp[jj]
            if not lat_temp[jj].data==0:
               lat_p_beached_SL_Cop[ii] = lat_temp[jj]
                
lon_p_beached_India_Cop = np.zeros(len(lon_p_India_Cop))
lat_p_beached_India_Cop = np.zeros(len(lat_p_India_Cop))
for ii in range(len(lon_p_India_Cop)):
    lon_temp = lon_p_India_Cop[ii,:]
    lat_temp = lat_p_India_Cop[ii,:]
    beached_temp = beached_p_India_Cop[ii,:]
    for jj in range(len(lon_temp)):
        if beached_temp[jj]==1:
            if not lon_temp[jj].data==0:
               lon_p_beached_India_Cop[ii] = lon_temp[jj]
            if not lat_temp[jj].data==0:
               lat_p_beached_India_Cop[ii] = lat_temp[jj]

lon_p_beached_Bang_Cop = np.zeros(len(lon_p_Bang_Cop))
lat_p_beached_Bang_Cop = np.zeros(len(lat_p_Bang_Cop))
for ii in range(len(lon_p_Bang_Cop)):
    lon_temp = lon_p_Bang_Cop[ii,:]
    lat_temp = lat_p_Bang_Cop[ii,:]
    beached_temp = beached_p_Bang_Cop[ii,:]
    for jj in range(len(lon_temp)):
        if beached_temp[jj]==1:
            if not lon_temp[jj].data==0:
               lon_p_beached_Bang_Cop[ii] = lon_temp[jj]
            if not lat_temp[jj].data==0:
               lat_p_beached_Bang_Cop[ii] = lat_temp[jj]

lon_p_beached_Myan_Cop = np.zeros(len(lon_p_Myan_Cop))
lat_p_beached_Myan_Cop = np.zeros(len(lat_p_Myan_Cop))
for ii in range(len(lon_p_Myan_Cop)):
    lon_temp = lon_p_Myan_Cop[ii,:]
    lat_temp = lat_p_Myan_Cop[ii,:]
    beached_temp = beached_p_Myan_Cop[ii,:]
    for jj in range(len(lon_temp)):
        if beached_temp[jj]==1:
            if not lon_temp[jj].data==0:
               lon_p_beached_Myan_Cop[ii] = lon_temp[jj]
            if not lat_temp[jj].data==0:
               lat_p_beached_Myan_Cop[ii] = lat_temp[jj]

lon_p_beached_Thai_Cop = np.zeros(len(lon_p_Thai_Cop))
lat_p_beached_Thai_Cop = np.zeros(len(lat_p_Thai_Cop))
for ii in range(len(lon_p_Thai_Cop)):
    lon_temp = lon_p_Thai_Cop[ii,:]
    lat_temp = lat_p_Thai_Cop[ii,:]
    beached_temp = beached_p_Thai_Cop[ii,:]
    for jj in range(len(lon_temp)):
        if beached_temp[jj]==1:
            if not lon_temp[jj].data==0:
               lon_p_beached_Thai_Cop[ii] = lon_temp[jj]
            if not lat_temp[jj].data==0:
               lat_p_beached_Thai_Cop[ii] = lat_temp[jj]
    
lon_p_beached_Indo_Cop = np.zeros(len(lon_p_Indo_Cop))
lat_p_beached_Indo_Cop = np.zeros(len(lat_p_Indo_Cop))
for ii in range(len(lon_p_Indo_Cop)):
    lon_temp = lon_p_Indo_Cop[ii,:]
    lat_temp = lat_p_Indo_Cop[ii,:]
    beached_temp = beached_p_Indo_Cop[ii,:]
    for jj in range(len(lon_temp)):
        if beached_temp[jj]==1:
            if not lon_temp[jj].data==0:
               lon_p_beached_Indo_Cop[ii] = lon_temp[jj]
            if not lat_temp[jj].data==0:
               lat_p_beached_Indo_Cop[ii] = lat_temp[jj]

In [ ]:
lon_p_beached_SL_ROMS = np.zeros(len(lon_p_SL_ROMS))
lat_p_beached_SL_ROMS = np.zeros(len(lat_p_SL_ROMS))
for ii in range(len(lon_p_SL_ROMS)):
    lon_temp = lon_p_SL_ROMS[ii,:]
    lat_temp = lat_p_SL_ROMS[ii,:]
    beached_temp = beached_p_SL_ROMS[ii,:]
    for jj in range(len(lon_temp)):
        if beached_temp[jj]==1:
            if not lon_temp[jj].data==0:
               lon_p_beached_SL_ROMS[ii] = lon_temp[jj]
            if not lat_temp[jj].data==0:
               lat_p_beached_SL_ROMS[ii] = lat_temp[jj]
                
lon_p_beached_India_ROMS = np.zeros(len(lon_p_India_ROMS))
lat_p_beached_India_ROMS = np.zeros(len(lat_p_India_ROMS))
for ii in range(len(lon_p_India_ROMS)):
    lon_temp = lon_p_India_ROMS[ii,:]
    lat_temp = lat_p_India_ROMS[ii,:]
    beached_temp = beached_p_India_ROMS[ii,:]
    for jj in range(len(lon_temp)):
        if beached_temp[jj]==1:
            if not lon_temp[jj].data==0:
               lon_p_beached_India_ROMS[ii] = lon_temp[jj]
            if not lat_temp[jj].data==0:
               lat_p_beached_India_ROMS[ii] = lat_temp[jj]

lon_p_beached_Bang_ROMS = np.zeros(len(lon_p_Bang_ROMS))
lat_p_beached_Bang_ROMS = np.zeros(len(lat_p_Bang_ROMS))
for ii in range(len(lon_p_Bang_ROMS)):
    lon_temp = lon_p_Bang_ROMS[ii,:]
    lat_temp = lat_p_Bang_ROMS[ii,:]
    beached_temp = beached_p_Bang_ROMS[ii,:]
    for jj in range(len(lon_temp)):
        if beached_temp[jj]==1:
            if not lon_temp[jj].data==0:
               lon_p_beached_Bang_ROMS[ii] = lon_temp[jj]
            if not lat_temp[jj].data==0:
               lat_p_beached_Bang_ROMS[ii] = lat_temp[jj]

lon_p_beached_Myan_ROMS = np.zeros(len(lon_p_Myan_ROMS))
lat_p_beached_Myan_ROMS = np.zeros(len(lat_p_Myan_ROMS))
for ii in range(len(lon_p_Myan_ROMS)):
    lon_temp = lon_p_Myan_ROMS[ii,:]
    lat_temp = lat_p_Myan_ROMS[ii,:]
    beached_temp = beached_p_Myan_ROMS[ii,:]
    for jj in range(len(lon_temp)):
        if beached_temp[jj]==1:
            if not lon_temp[jj].data==0:
               lon_p_beached_Myan_ROMS[ii] = lon_temp[jj]
            if not lat_temp[jj].data==0:
               lat_p_beached_Myan_ROMS[ii] = lat_temp[jj]

lon_p_beached_Thai_ROMS = np.zeros(len(lon_p_Thai_ROMS))
lat_p_beached_Thai_ROMS = np.zeros(len(lat_p_Thai_ROMS))
for ii in range(len(lon_p_Thai_ROMS)):
    lon_temp = lon_p_Thai_ROMS[ii,:]
    lat_temp = lat_p_Thai_ROMS[ii,:]
    beached_temp = beached_p_Thai_ROMS[ii,:]
    for jj in range(len(lon_temp)):
        if beached_temp[jj]==1:
            if not lon_temp[jj].data==0:
               lon_p_beached_Thai_ROMS[ii] = lon_temp[jj]
            if not lat_temp[jj].data==0:
               lat_p_beached_Thai_ROMS[ii] = lat_temp[jj]
    
lon_p_beached_Indo_ROMS = np.zeros(len(lon_p_Indo_ROMS))
lat_p_beached_Indo_ROMS = np.zeros(len(lat_p_Indo_ROMS))
for ii in range(len(lon_p_Indo_ROMS)):
    lon_temp = lon_p_Indo_ROMS[ii,:]
    lat_temp = lat_p_Indo_ROMS[ii,:]
    beached_temp = beached_p_Indo_ROMS[ii,:]
    for jj in range(len(lon_temp)):
        if beached_temp[jj]==1:
            if not lon_temp[jj].data==0:
               lon_p_beached_Indo_ROMS[ii] = lon_temp[jj]
            if not lat_temp[jj].data==0:
               lat_p_beached_Indo_ROMS[ii] = lat_temp[jj]


In [ ]:
lon_p_beached_SL_Cop[lon_p_beached_SL_Cop==0] = np.nan
lat_p_beached_SL_Cop[lat_p_beached_SL_Cop==0] = np.nan
lon_p_beached_India_Cop[lon_p_beached_India_Cop==0] = np.nan
lat_p_beached_India_Cop[lat_p_beached_India_Cop==0] = np.nan
lon_p_beached_Bang_Cop[lon_p_beached_Bang_Cop==0] = np.nan
lat_p_beached_Bang_Cop[lat_p_beached_Bang_Cop==0] = np.nan
lon_p_beached_Myan_Cop[lon_p_beached_Myan_Cop==0] = np.nan
lat_p_beached_Myan_Cop[lat_p_beached_Myan_Cop==0] = np.nan
lon_p_beached_Thai_Cop[lon_p_beached_Thai_Cop==0] = np.nan
lat_p_beached_Thai_Cop[lat_p_beached_Thai_Cop==0] = np.nan
lon_p_beached_Indo_Cop[lon_p_beached_Indo_Cop==0] = np.nan
lat_p_beached_Indo_Cop[lat_p_beached_Indo_Cop==0] = np.nan

In [ ]:
lon_p_beached_SL_ROMS[lon_p_beached_SL_ROMS==0] = np.nan
lat_p_beached_SL_ROMS[lat_p_beached_SL_ROMS==0] = np.nan
lon_p_beached_India_ROMS[lon_p_beached_India_ROMS==0] = np.nan
lat_p_beached_India_ROMS[lat_p_beached_India_ROMS==0] = np.nan
lon_p_beached_Bang_ROMS[lon_p_beached_Bang_ROMS==0] = np.nan
lat_p_beached_Bang_ROMS[lat_p_beached_Bang_ROMS==0] = np.nan
lon_p_beached_Myan_ROMS[lon_p_beached_Myan_ROMS==0] = np.nan
lat_p_beached_Myan_ROMS[lat_p_beached_Myan_ROMS==0] = np.nan
lon_p_beached_Thai_ROMS[lon_p_beached_Thai_ROMS==0] = np.nan
lat_p_beached_Thai_ROMS[lat_p_beached_Thai_ROMS==0] = np.nan
lon_p_beached_Indo_ROMS[lon_p_beached_Indo_ROMS==0] = np.nan
lat_p_beached_Indo_ROMS[lat_p_beached_Indo_ROMS==0] = np.nan

In [ ]:
# create figure
fig = plt.figure(figsize=(20, 10), dpi=300)
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([77,99,3.5,24]) # full domain
ax.add_feature(cartopy.feature.BORDERS, linestyle='-', linewidth=1.0)
gl = ax.gridlines(draw_labels=True)
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}

transform = ccrs.PlateCarree()._as_mpl_transform(ax)

m1 = ax.pcolormesh(lon_grid_u,lat_grid_u,speed_subset54, cmap='Blues_r', vmin=np.amin(speed_subset54), vmax=np.amax(speed_subset54)) # mappable content

p0 = ax.scatter(lon_p_beached_SL_Cop[:], lat_p_beached_SL_Cop[:],color='lightpink', s=5, marker='s')
p1 = ax.scatter(lon_p_beached_India_Cop[:], lat_p_beached_India_Cop[:],color='gold', s=5)
p2 = ax.scatter(lon_p_beached_Bang_Cop[:], lat_p_beached_Bang_Cop[:],color='crimson', s=5, marker='v')
p3 = ax.scatter(lon_p_beached_Myan_Cop[:], lat_p_beached_Myan_Cop[:],color='mediumaquamarine', s=5, marker='*')
p4 = ax.scatter(lon_p_beached_Thai_Cop[:], lat_p_beached_Thai_Cop[:],color='k', s=5, marker='p')
p5 = ax.scatter(lon_p_beached_Indo_Cop[:], lat_p_beached_Indo_Cop[:],color='cyan', s=5, marker='d')

plt.legend((p0, p1, p2, p3, p4, p5),
           ('Sri Lanka', 'India', 'Bangladesh', 'Myanmar', 'Thailand', 'Indonesia'),
           scatterpoints=5,
           loc='upper left',
           ncol=1,
           fontsize=12)

fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/BoB_CMEMS_beached", bbox_inches='tight')


In [ ]:
# create figure
fig = plt.figure(figsize=(20, 10), dpi=300)
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([77,99,3.5,24]) # full domain
ax.add_feature(cartopy.feature.BORDERS, linestyle='-', linewidth=1.0)
gl = ax.gridlines(draw_labels=True)
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}

transform = ccrs.PlateCarree()._as_mpl_transform(ax)

m2 = ax.pcolormesh(lon_grid_uR,lat_grid_uR,speed_subset54R, cmap='Blues_r', vmin=np.amin(speed_subset54R), vmax=np.amax(speed_subset54R)) # mappable content

ax.scatter(lon_p_beached_SL_ROMS[:], lat_p_beached_SL_ROMS[:],color='lightpink', s=5, marker='s')
ax.scatter(lon_p_beached_India_ROMS[:], lat_p_beached_India_ROMS[:],color='gold', s=5)
ax.scatter(lon_p_beached_Bang_ROMS[:], lat_p_beached_Bang_ROMS[:],color='crimson', s=5, marker='v')
ax.scatter(lon_p_beached_Myan_ROMS[:], lat_p_beached_Myan_ROMS[:],color='mediumaquamarine', s=5, marker='*')
ax.scatter(lon_p_beached_Thai_ROMS[:], lat_p_beached_Thai_ROMS[:],color='k', s=5, marker='p')
ax.scatter(lon_p_beached_Indo_ROMS[:], lat_p_beached_Indo_ROMS[:],color='cyan', s=5, marker='d')

fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/BoB_ROMS_beached", bbox_inches='tight')


In [ ]:
# create figure
fig = plt.figure(figsize=(20, 10), dpi=300)
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([93,99,3.5,9])
ax.add_feature(cartopy.feature.BORDERS, linestyle='-', linewidth=1.0)
gl = ax.gridlines(draw_labels=True)
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}

transform = ccrs.PlateCarree()._as_mpl_transform(ax)

m2 = ax.pcolormesh(lon_grid_u,lat_grid_u,speed_subset54, cmap='Blues_r', vmin=np.amin(speed_subset54), vmax=np.amax(speed_subset54)) # mappable content

ax.scatter(lon_p_beached_SL_Cop[:], lat_p_beached_SL_Cop[:],color='lightpink', s=5, marker='s')
ax.scatter(lon_p_beached_India_Cop[:], lat_p_beached_India_Cop[:],color='gold', s=5)
ax.scatter(lon_p_beached_Bang_Cop[:], lat_p_beached_Bang_Cop[:],color='crimson', s=5, marker='v')
ax.scatter(lon_p_beached_Myan_Cop[:], lat_p_beached_Myan_Cop[:],color='mediumaquamarine', s=5, marker='*')
ax.scatter(lon_p_beached_Thai_Cop[:], lat_p_beached_Thai_Cop[:],color='k', s=5, marker='p')
ax.scatter(lon_p_beached_Indo_Cop[:], lat_p_beached_Indo_Cop[:],color='cyan', s=5, marker='d')

cbar = fig.colorbar(m2, ax=ax,location='bottom', cmap='Blues')

# Get the current position of the colorbar
pos = cbar.ax.get_position()

# Set the new position of the colorbar
cbar.ax.set_position([pos.x0+0.25, pos.y0+0.075, pos.width-0.5, pos.height])
cbar.set_label('m/s', fontsize=12, x=1.04, labelpad=-32)

fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/BoB_CMEMS_beached_zoomed_indo", bbox_inches='tight')


In [ ]:
# create figure
fig = plt.figure(figsize=(20, 10), dpi=300)
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([93,99,3.5,9])
ax.add_feature(cartopy.feature.BORDERS, linestyle='-', linewidth=1.0)
gl = ax.gridlines(draw_labels=True)
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}

transform = ccrs.PlateCarree()._as_mpl_transform(ax)

m2 = ax.pcolormesh(lon_grid_uR,lat_grid_uR,speed_subset54R, cmap='Blues_r', vmin=np.amin(speed_subset54R), vmax=np.amax(speed_subset54R)) # mappable content

ax.scatter(lon_p_beached_SL_ROMS[:], lat_p_beached_SL_ROMS[:],color='lightpink', s=5, marker='s')
ax.scatter(lon_p_beached_India_ROMS[:], lat_p_beached_India_ROMS[:],color='gold', s=5)
ax.scatter(lon_p_beached_Bang_ROMS[:], lat_p_beached_Bang_ROMS[:],color='crimson', s=5, marker='v')
ax.scatter(lon_p_beached_Myan_ROMS[:], lat_p_beached_Myan_ROMS[:],color='mediumaquamarine', s=5, marker='*')
ax.scatter(lon_p_beached_Thai_ROMS[:], lat_p_beached_Thai_ROMS[:],color='k', s=5, marker='p')
ax.scatter(lon_p_beached_Indo_ROMS[:], lat_p_beached_Indo_ROMS[:],color='cyan', s=5, marker='d')

cbar = fig.colorbar(m2, ax=ax,location='bottom', cmap='Blues')

# Get the current position of the colorbar
pos = cbar.ax.get_position()

# Set the new position of the colorbar
cbar.ax.set_position([pos.x0+0.25, pos.y0+0.075, pos.width-0.5, pos.height])
cbar.set_label('m/s', fontsize=12, x=1.04, labelpad=-32)

fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/BoB_ROMS_beached_zoomed_indo", bbox_inches='tight')


In [ ]:
# Fig B1

In [ ]:
#connectivity matrices
Cop_hourly_July2020 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_Cop_hourly_July2020.csv'
Cop_daily_July2020 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_Cop_daily_July2020.csv'
ROMS_hourly_July2020 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_ROMS_hourly_July2020.csv'
ROMS_daily_July2020 = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_ROMS_daily_July2020.csv'


In [ ]:
input_conn_mat_file_monsoon_CMEMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_Cop_daily_Jun2018-Sept2019_monsoon_seasononly.csv'
input_conn_mat_file_monsoon_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_ROMS_daily_Jun2018-Sept2019_monsoon_seasononly.csv'
input_conn_mat_file_post_CMEMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_Cop_daily_Oct2018-Sept2019_postmonsoon_seasononly.csv'
input_conn_mat_file_post_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_ROMS_daily_Oct2018-Sept2019_postmonsoon_seasononly.csv'
input_conn_mat_file_pre_CMEMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_Cop_daily_Feb-Sept2019_premonsoon_seasononly.csv'
input_conn_mat_file_pre_ROMS = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/connectivity/conn_mat6x7_norm_BoB_uniform_ROMS_daily_Feb-Sept2019_premonsoon_seasononly.csv'

In [ ]:
# velocity files
velocity_input_file_Chourly = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/ocean_velocities_July2020_hourly.nc'
velocity_input_file_Cdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/ocean_velocities_July2020_daily.nc'

velocity_input_file_Rhourly = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/ocean_velocities_ROMS_July2020_hourly.nc'
velocity_u_input_file_Rdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/ocean_velocities_u_ROMS_July2020_daily.nc'
velocity_v_input_file_Rdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/input/processed/paper_data/ocean_velocities_v_ROMS_July2020_daily.nc'


In [ ]:
# particle files
input_pfile_SL_Chourly = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_SL_uniform_Cop_hourly_July2020.nc'
input_pfile_India_Chourly = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_India_uniform_Cop_hourly_July2020.nc'
input_pfile_Bang_Chourly = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Bang_uniform_Cop_hourly_July2020.nc'
input_pfile_Myan_Chourly = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Myan_uniform_Cop_hourly_July2020.nc'
input_pfile_Thai_Chourly = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Thai_uniform_Cop_hourly_July2020.nc'
input_pfile_Indo_Chourly = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Indonesia_uniform_Cop_hourly_July2020.nc'

input_pfile_SL_Cdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_SL_uniform_Cop_daily_July2020.nc'
input_pfile_India_Cdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_India_uniform_Cop_daily_July2020.nc'
input_pfile_Bang_Cdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Bang_uniform_Cop_daily_July2020.nc'
input_pfile_Myan_Cdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Myan_uniform_Cop_daily_July2020.nc'
input_pfile_Thai_Cdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Thai_uniform_Cop_daily_July2020.nc'
input_pfile_Indo_Cdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Indonesia_uniform_Cop_daily_July2020.nc'

input_pfile_SL_Rhourly = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_SL_uniform_ROMS_hourly_July2020.nc'
input_pfile_India_Rhourly = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_India_uniform_ROMS_hourly_July2020.nc'
input_pfile_Bang_Rhourly = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Bang_uniform_ROMS_hourly_July2020.nc'
input_pfile_Myan_Rhourly = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Myan_uniform_ROMS_hourly_July2020.nc'
input_pfile_Thai_Rhourly = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Thai_uniform_ROMS_hourly_July2020.nc'
input_pfile_Indo_Rhourly = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Indonesia_uniform_ROMS_hourly_July2020.nc'

input_pfile_SL_Rdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_SL_uniform_ROMS_daily_July2020.nc'
input_pfile_India_Rdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_India_uniform_ROMS_daily_July2020.nc'
input_pfile_Bang_Rdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Bang_uniform_ROMS_daily_July2020.nc'
input_pfile_Myan_Rdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Myan_uniform_ROMS_daily_July2020.nc'
input_pfile_Thai_Rdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Thai_uniform_ROMS_daily_July2020.nc'
input_pfile_Indo_Rdaily = '/gpfs/home/rpe16nbu/projects/ocpp_mo1/data/output/paper_data/BoB_Indonesia_uniform_ROMS_daily_July2020.nc'


In [ ]:
openfile_p9_CH = Dataset(input_pfile_SL_Chourly)
lon_p_SL_CH = openfile_p9_CH.variables['lon']
lat_p_SL_CH = openfile_p9_CH.variables['lat']
time_p_SL_CH = openfile_p9_CH.variables['time']
openfile_p10_CH = Dataset(input_pfile_India_Chourly)
lon_p_India_CH = openfile_p10_CH.variables['lon']
lat_p_India_CH = openfile_p10_CH.variables['lat']
openfile_p11_CH = Dataset(input_pfile_Bang_Chourly)
lon_p_Bang_CH = openfile_p11_CH.variables['lon']
lat_p_Bang_CH = openfile_p11_CH.variables['lat']
openfile_p12_CH = Dataset(input_pfile_Myan_Chourly)
lon_p_Myan_CH = openfile_p12_CH.variables['lon']
lat_p_Myan_CH = openfile_p12_CH.variables['lat']
openfile_p13_CH = Dataset(input_pfile_Thai_Chourly)
lon_p_Thai_CH = openfile_p13_CH.variables['lon']
lat_p_Thai_CH = openfile_p13_CH.variables['lat']
openfile_p14_CH = Dataset(input_pfile_Indo_Chourly)
lon_p_Indo_CH = openfile_p14_CH.variables['lon']
lat_p_Indo_CH = openfile_p14_CH.variables['lat']
        
lon_p_end_SL_CH = np.zeros(len(lon_p_SL_CH))
lat_p_end_SL_CH = np.zeros(len(lat_p_SL_CH))
for ii in range(len(lon_p_SL_CH)):
    lon_temp = lon_p_SL_CH[ii,:]
    lat_temp = lat_p_SL_CH[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
                lon_p_end_SL_CH[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
                lat_p_end_SL_CH[ii] = lat_temp[jj]
                
lon_p_end_India_CH = np.zeros(len(lon_p_India_CH))
lat_p_end_India_CH = np.zeros(len(lat_p_India_CH))
for ii in range(len(lon_p_India_CH)):
    lon_temp = lon_p_India_CH[ii,:]
    lat_temp = lat_p_India_CH[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_India_CH[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_India_CH[ii] = lat_temp[jj]

lon_p_end_Bang_CH = np.zeros(len(lon_p_Bang_CH))
lat_p_end_Bang_CH = np.zeros(len(lat_p_Bang_CH))
for ii in range(len(lon_p_Bang_CH)):
    lon_temp = lon_p_Bang_CH[ii,:]
    lat_temp = lat_p_Bang_CH[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Bang_CH[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Bang_CH[ii] = lat_temp[jj]

lon_p_end_Myan_CH = np.zeros(len(lon_p_Myan_CH))
lat_p_end_Myan_CH = np.zeros(len(lat_p_Myan_CH))
for ii in range(len(lon_p_Myan_CH)):
    lon_temp = lon_p_Myan_CH[ii,:]
    lat_temp = lat_p_Myan_CH[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Myan_CH[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Myan_CH[ii] = lat_temp[jj]

lon_p_end_Thai_CH = np.zeros(len(lon_p_Thai_CH))
lat_p_end_Thai_CH = np.zeros(len(lat_p_Thai_CH))
for ii in range(len(lon_p_Thai_CH)):
    lon_temp = lon_p_Thai_CH[ii,:]
    lat_temp = lat_p_Thai_CH[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Thai_CH[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Thai_CH[ii] = lat_temp[jj]
    
lon_p_end_Indo_CH = np.zeros(len(lon_p_Indo_CH))
lat_p_end_Indo_CH = np.zeros(len(lat_p_Indo_CH))
for ii in range(len(lon_p_Indo_CH)):
    lon_temp = lon_p_Indo_CH[ii,:]
    lat_temp = lat_p_Indo_CH[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Indo_CH[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Indo_CH[ii] = lat_temp[jj]


In [ ]:
openfile_p9_CD = Dataset(input_pfile_SL_Cdaily)
lon_p_SL_CD = openfile_p9_CD.variables['lon']
lat_p_SL_CD = openfile_p9_CD.variables['lat']
time_p_SL_CD = openfile_p9_CD.variables['time']
openfile_p10_CD = Dataset(input_pfile_India_Cdaily)
lon_p_India_CD = openfile_p10_CD.variables['lon']
lat_p_India_CD = openfile_p10_CD.variables['lat']
openfile_p11_CD = Dataset(input_pfile_Bang_Cdaily)
lon_p_Bang_CD = openfile_p11_CD.variables['lon']
lat_p_Bang_CD = openfile_p11_CD.variables['lat']
openfile_p12_CD = Dataset(input_pfile_Myan_Cdaily)
lon_p_Myan_CD = openfile_p12_CD.variables['lon']
lat_p_Myan_CD = openfile_p12_CD.variables['lat']
openfile_p13_CD = Dataset(input_pfile_Thai_Cdaily)
lon_p_Thai_CD = openfile_p13_CD.variables['lon']
lat_p_Thai_CD = openfile_p13_CD.variables['lat']
openfile_p14_CD = Dataset(input_pfile_Indo_Cdaily)
lon_p_Indo_CD = openfile_p14_CD.variables['lon']
lat_p_Indo_CD = openfile_p14_CD.variables['lat']
        
lon_p_end_SL_CD = np.zeros(len(lon_p_SL_CD))
lat_p_end_SL_CD = np.zeros(len(lat_p_SL_CD))
for ii in range(len(lon_p_SL_CD)):
    lon_temp = lon_p_SL_CD[ii,:]
    lat_temp = lat_p_SL_CD[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
                lon_p_end_SL_CD[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
                lat_p_end_SL_CD[ii] = lat_temp[jj]
                
lon_p_end_India_CD = np.zeros(len(lon_p_India_CD))
lat_p_end_India_CD = np.zeros(len(lat_p_India_CD))
for ii in range(len(lon_p_India_CD)):
    lon_temp = lon_p_India_CD[ii,:]
    lat_temp = lat_p_India_CD[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_India_CD[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_India_CD[ii] = lat_temp[jj]

lon_p_end_Bang_CD = np.zeros(len(lon_p_Bang_CD))
lat_p_end_Bang_CD = np.zeros(len(lat_p_Bang_CD))
for ii in range(len(lon_p_Bang_CD)):
    lon_temp = lon_p_Bang_CD[ii,:]
    lat_temp = lat_p_Bang_CD[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Bang_CD[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Bang_CD[ii] = lat_temp[jj]

lon_p_end_Myan_CD = np.zeros(len(lon_p_Myan_CD))
lat_p_end_Myan_CD = np.zeros(len(lat_p_Myan_CD))
for ii in range(len(lon_p_Myan_CD)):
    lon_temp = lon_p_Myan_CD[ii,:]
    lat_temp = lat_p_Myan_CD[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Myan_CD[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Myan_CD[ii] = lat_temp[jj]

lon_p_end_Thai_CD = np.zeros(len(lon_p_Thai_CD))
lat_p_end_Thai_CD = np.zeros(len(lat_p_Thai_CD))
for ii in range(len(lon_p_Thai_CD)):
    lon_temp = lon_p_Thai_CD[ii,:]
    lat_temp = lat_p_Thai_CD[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Thai_CD[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Thai_CD[ii] = lat_temp[jj]
    
lon_p_end_Indo_CD = np.zeros(len(lon_p_Indo_CD))
lat_p_end_Indo_CD = np.zeros(len(lat_p_Indo_CD))
for ii in range(len(lon_p_Indo_CD)):
    lon_temp = lon_p_Indo_CD[ii,:]
    lat_temp = lat_p_Indo_CD[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Indo_CD[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Indo_CD[ii] = lat_temp[jj]



In [ ]:
openfile_p9_RD = Dataset(input_pfile_SL_Rdaily)
lon_p_SL_RD = openfile_p9_RD.variables['lon']
lat_p_SL_RD = openfile_p9_RD.variables['lat']
time_p_SL_RD = openfile_p9_RD.variables['time']
openfile_p10_RD = Dataset(input_pfile_India_Rdaily)
lon_p_India_RD = openfile_p10_RD.variables['lon']
lat_p_India_RD = openfile_p10_RD.variables['lat']
openfile_p11_RD = Dataset(input_pfile_Bang_Rdaily)
lon_p_Bang_RD = openfile_p11_RD.variables['lon']
lat_p_Bang_RD = openfile_p11_RD.variables['lat']
openfile_p12_RD = Dataset(input_pfile_Myan_Rdaily)
lon_p_Myan_RD = openfile_p12_RD.variables['lon']
lat_p_Myan_RD = openfile_p12_RD.variables['lat']
openfile_p13_RD = Dataset(input_pfile_Thai_Rdaily)
lon_p_Thai_RD = openfile_p13_RD.variables['lon']
lat_p_Thai_RD = openfile_p13_RD.variables['lat']
openfile_p14_RD = Dataset(input_pfile_Indo_Rdaily)
lon_p_Indo_RD = openfile_p14_RD.variables['lon']
lat_p_Indo_RD = openfile_p14_RD.variables['lat']
        
lon_p_end_SL_RD = np.zeros(len(lon_p_SL_RD))
lat_p_end_SL_RD = np.zeros(len(lat_p_SL_RD))
for ii in range(len(lon_p_SL_RD)):
    lon_temp = lon_p_SL_RD[ii,:]
    lat_temp = lat_p_SL_RD[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
                lon_p_end_SL_RD[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
                lat_p_end_SL_RD[ii] = lat_temp[jj]
                
lon_p_end_India_RD = np.zeros(len(lon_p_India_RD))
lat_p_end_India_RD = np.zeros(len(lat_p_India_RD))
for ii in range(len(lon_p_India_RD)):
    lon_temp = lon_p_India_RD[ii,:]
    lat_temp = lat_p_India_RD[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_India_RD[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_India_RD[ii] = lat_temp[jj]

lon_p_end_Bang_RD = np.zeros(len(lon_p_Bang_RD))
lat_p_end_Bang_RD = np.zeros(len(lat_p_Bang_RD))
for ii in range(len(lon_p_Bang_RD)):
    lon_temp = lon_p_Bang_RD[ii,:]
    lat_temp = lat_p_Bang_RD[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Bang_RD[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Bang_RD[ii] = lat_temp[jj]

lon_p_end_Myan_RD = np.zeros(len(lon_p_Myan_RD))
lat_p_end_Myan_RD = np.zeros(len(lat_p_Myan_RD))
for ii in range(len(lon_p_Myan_RD)):
    lon_temp = lon_p_Myan_RD[ii,:]
    lat_temp = lat_p_Myan_RD[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Myan_RD[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Myan_RD[ii] = lat_temp[jj]

lon_p_end_Thai_RD = np.zeros(len(lon_p_Thai_RD))
lat_p_end_Thai_RD = np.zeros(len(lat_p_Thai_RD))
for ii in range(len(lon_p_Thai_RD)):
    lon_temp = lon_p_Thai_RD[ii,:]
    lat_temp = lat_p_Thai_RD[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Thai_RD[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Thai_RD[ii] = lat_temp[jj]
    
lon_p_end_Indo_RD = np.zeros(len(lon_p_Indo_RD))
lat_p_end_Indo_RD = np.zeros(len(lat_p_Indo_RD))
for ii in range(len(lon_p_Indo_RD)):
    lon_temp = lon_p_Indo_RD[ii,:]
    lat_temp = lat_p_Indo_RD[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Indo_RD[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Indo_RD[ii] = lat_temp[jj]


In [ ]:
openfile_p9_RH = Dataset(input_pfile_SL_Rhourly)
lon_p_SL_RH = openfile_p9_RH.variables['lon']
lat_p_SL_RH = openfile_p9_RH.variables['lat']
time_p_SL_RH = openfile_p9_RH.variables['time']
openfile_p10_RH = Dataset(input_pfile_India_Rhourly)
lon_p_India_RH = openfile_p10_RH.variables['lon']
lat_p_India_RH = openfile_p10_RH.variables['lat']
openfile_p11_RH = Dataset(input_pfile_Bang_Rhourly)
lon_p_Bang_RH = openfile_p11_RH.variables['lon']
lat_p_Bang_RH = openfile_p11_RH.variables['lat']
openfile_p12_RH = Dataset(input_pfile_Myan_Rhourly)
lon_p_Myan_RH = openfile_p12_RH.variables['lon']
lat_p_Myan_RH = openfile_p12_RH.variables['lat']
openfile_p13_RH = Dataset(input_pfile_Thai_Rhourly)
lon_p_Thai_RH = openfile_p13_RH.variables['lon']
lat_p_Thai_RH = openfile_p13_RH.variables['lat']
openfile_p14_RH = Dataset(input_pfile_Indo_Rhourly)
lon_p_Indo_RH = openfile_p14_RH.variables['lon']
lat_p_Indo_RH = openfile_p14_RH.variables['lat']
        
lon_p_end_SL_RH = np.zeros(len(lon_p_SL_RH))
lat_p_end_SL_RH = np.zeros(len(lat_p_SL_RH))
for ii in range(len(lon_p_SL_RH)):
    lon_temp = lon_p_SL_RH[ii,:]
    lat_temp = lat_p_SL_RH[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
                lon_p_end_SL_RH[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
                lat_p_end_SL_RH[ii] = lat_temp[jj]
                
lon_p_end_India_RH = np.zeros(len(lon_p_India_RH))
lat_p_end_India_RH = np.zeros(len(lat_p_India_RH))
for ii in range(len(lon_p_India_RH)):
    lon_temp = lon_p_India_RH[ii,:]
    lat_temp = lat_p_India_RH[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_India_RH[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_India_RH[ii] = lat_temp[jj]

lon_p_end_Bang_RH = np.zeros(len(lon_p_Bang_RH))
lat_p_end_Bang_RH = np.zeros(len(lat_p_Bang_RH))
for ii in range(len(lon_p_Bang_RH)):
    lon_temp = lon_p_Bang_RH[ii,:]
    lat_temp = lat_p_Bang_RH[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Bang_RH[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Bang_RH[ii] = lat_temp[jj]

lon_p_end_Myan_RH = np.zeros(len(lon_p_Myan_RH))
lat_p_end_Myan_RH = np.zeros(len(lat_p_Myan_RH))
for ii in range(len(lon_p_Myan_RH)):
    lon_temp = lon_p_Myan_RH[ii,:]
    lat_temp = lat_p_Myan_RH[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Myan_RH[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Myan_RH[ii] = lat_temp[jj]

lon_p_end_Thai_RH = np.zeros(len(lon_p_Thai_RH))
lat_p_end_Thai_RH = np.zeros(len(lat_p_Thai_RH))
for ii in range(len(lon_p_Thai_RH)):
    lon_temp = lon_p_Thai_RH[ii,:]
    lat_temp = lat_p_Thai_RH[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Thai_RH[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Thai_RH[ii] = lat_temp[jj]
    
lon_p_end_Indo_RH = np.zeros(len(lon_p_Indo_RH))
lat_p_end_Indo_RH = np.zeros(len(lat_p_Indo_RH))
for ii in range(len(lon_p_Indo_RH)):
    lon_temp = lon_p_Indo_RH[ii,:]
    lat_temp = lat_p_Indo_RH[ii,:]
    for jj in range(len(lon_temp)):
        if not lon_temp[jj].data==0:
               lon_p_end_Indo_RH[ii] = lon_temp[jj]
        if not lat_temp[jj].data==0:
               lat_p_end_Indo_RH[ii] = lat_temp[jj]


In [ ]:
vminR = 0
vmaxR = 2.25

In [ ]:
def plot_particle_end_locs_CMEMS_noextras(ax, velocity_input_file, lon_p_end_SL, lon_p_end_India, lon_p_end_Bang, 
                                    lon_p_end_Myan, lon_p_end_Thai, lon_p_end_Indo, lat_p_end_SL, lat_p_end_India, 
                                          lat_p_end_Bang, lat_p_end_Myan, lat_p_end_Thai, lat_p_end_Indo, vmin, vmax):
    
    openfile_u = Dataset(velocity_input_file)
    lon_vec_u = openfile_u.variables['longitude']
    lat_vec_u = openfile_u.variables['latitude']
    time_u = openfile_u.variables['time']
    u =  openfile_u.variables['uo']
    v =  openfile_u.variables['vo']

    u_subset_start = u[0,0,:,:]
    u_subset_end= u[-1,0,:,:]
    v_subset_start = v[0,0,:,:]
    v_subset_end = v[-1,0,:,:]
    speed_subset_start = np.sqrt(np.abs(u_subset_start)**2 + np.abs(v_subset_start)**2)
    speed_subset_end = np.sqrt(np.abs(u_subset_end)**2 + np.abs(v_subset_end)**2)

    # create grids of coordinates rather than just lists
    lon_grid_u, lat_grid_u = np.meshgrid(lon_vec_u, lat_vec_u)

    # defining datetime objects for start and end time of simulation
    interval_type = 'hours'
    interval_num_start = int(time_u[0])
    interval_num_end = int(time_u[-1])

    origin_time_u = datetime.datetime.strptime('1950/01/01 00:00:00','%Y/%m/%d %H:%M:%S') # simulation time output is number of hours since 00:00:00 01-01-1950
    start_time_u = origin_time_u + datetime.timedelta(**{interval_type: interval_num_start}) # turns integers into a datetime object to be added to origin_time
    end_time_u = origin_time_u + datetime.timedelta(**{interval_type: interval_num_end})
    
    # Normalize the data
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    
    m2 = ax.pcolormesh(lon_grid_u[:,:],lat_grid_u[:,:],speed_subset_end, cmap='Blues_r', norm=norm)#, vmin=-2, vmax =3) # mappable content
    p0 = ax.scatter(lon_p_end_SL[:], lat_p_end_SL[:],color='lightpink', s=0.01)
    p1 = ax.scatter(lon_p_end_India[:], lat_p_end_India[:],color='gold', s=0.01)
    p2 = ax.scatter(lon_p_end_Bang[:], lat_p_end_Bang[:],color='crimson', s=0.01)
    p3 = ax.scatter(lon_p_end_Myan[:], lat_p_end_Myan[:],color='mediumaquamarine', s=0.01)
    p4 = ax.scatter(lon_p_end_Thai[:], lat_p_end_Thai[:],color='k', s=0.01)
    p5 = ax.scatter(lon_p_end_Indo[:], lat_p_end_Indo[:],color='cyan', s=0.01)
    
    ax.xaxis.set_ticks_position('top')    
    if ax == axs[1,0]:
        ax.set_xticks([])
        ax.set_xticklabels([])

    ax.legend((p0, p1, p2, p3, p4, p5),
           ('Sri Lanka', 'India', 'Bangladesh', 'Myanmar', 'Thailand', 'Indonesia'),
           scatterpoints=5,
           loc='upper left',
           ncol=1,
           fontsize=8,
           markerscale=10)      
    
    return m2

In [ ]:
def plot_particle_end_locs_CMEMS_noextras1(ax, velocity_input_file, lon_p_end_SL, lon_p_end_India, lon_p_end_Bang, 
                                    lon_p_end_Myan, lon_p_end_Thai, lon_p_end_Indo, lat_p_end_SL, lat_p_end_India, 
                                          lat_p_end_Bang, lat_p_end_Myan, lat_p_end_Thai, lat_p_end_Indo, vmin, vmax):
    
    openfile_u = Dataset(velocity_input_file)
    lon_vec_u = openfile_u.variables['longitude']
    lat_vec_u = openfile_u.variables['latitude']
    time_u = openfile_u.variables['time']
    u =  openfile_u.variables['uo']
    v =  openfile_u.variables['vo']

    u_subset_start = u[0,0,:,:]
    u_subset_end= u[-1,0,:,:]
    v_subset_start = v[0,0,:,:]
    v_subset_end = v[-1,0,:,:]
    speed_subset_start = np.sqrt(np.abs(u_subset_start)**2 + np.abs(v_subset_start)**2)
    speed_subset_end = np.sqrt(np.abs(u_subset_end)**2 + np.abs(v_subset_end)**2)

    # create grids of coordinates rather than just lists
    lon_grid_u, lat_grid_u = np.meshgrid(lon_vec_u, lat_vec_u)

    # defining datetime objects for start and end time of simulation
    interval_type = 'hours'
    interval_num_start = int(time_u[0])
    interval_num_end = int(time_u[-1])

    origin_time_u = datetime.datetime.strptime('1950/01/01 00:00:00','%Y/%m/%d %H:%M:%S') # simulation time output is number of hours since 00:00:00 01-01-1950
    start_time_u = origin_time_u + datetime.timedelta(**{interval_type: interval_num_start}) # turns integers into a datetime object to be added to origin_time
    end_time_u = origin_time_u + datetime.timedelta(**{interval_type: interval_num_end})
    
    # Normalize the data
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    
    m2 = ax.pcolormesh(lon_grid_u[:,:],lat_grid_u[:,:],speed_subset_end, cmap='Blues_r', norm=norm)#, vmin=-2, vmax =3) # mappable content
    p0 = ax.scatter(lon_p_end_SL[:], lat_p_end_SL[:],color='lightpink', s=0.01)
    p1 = ax.scatter(lon_p_end_India[:], lat_p_end_India[:],color='gold', s=0.01)
    p2 = ax.scatter(lon_p_end_Bang[:], lat_p_end_Bang[:],color='crimson', s=0.01)
    p3 = ax.scatter(lon_p_end_Myan[:], lat_p_end_Myan[:],color='mediumaquamarine', s=0.01)
    p4 = ax.scatter(lon_p_end_Thai[:], lat_p_end_Thai[:],color='k', s=0.01)
    p5 = ax.scatter(lon_p_end_Indo[:], lat_p_end_Indo[:],color='cyan', s=0.01)
    
    ax.xaxis.set_ticks_position('top')    
    if ax == axs[1,0]:
        ax.set_xticks([])
        ax.set_xticklabels([])
  
    return m2

In [ ]:
def plot_particle_end_locs_ROMS_hourly_noextras(ax, velocity_input_file, lon_p_end_SL, lon_p_end_India, lon_p_end_Bang, 
                                    lon_p_end_Myan, lon_p_end_Thai, lon_p_end_Indo, lat_p_end_SL, lat_p_end_India, 
                                                lat_p_end_Bang, lat_p_end_Myan, lat_p_end_Thai, lat_p_end_Indo, vmin, vmax):
    
    openfile_u = Dataset(velocity_input_file)
    lon_vec_u = openfile_u.variables['lon_u']
    lat_vec_u = openfile_u.variables['lat_u']
    lon_vec_v = openfile_u.variables['lon_v']
    lat_vec_v = openfile_u.variables['lat_v']
    time_u = openfile_u.variables['ocean_time']
    u =  openfile_u.variables['u']
    v =  openfile_u.variables['v']
    
    u_subset_start = u[0,0,:-1,:]
    u_subset_end= u[-1,0,:-1,:]
    v_subset_start = v[0,0,:,:-1]
    v_subset_end = v[-1,0,:,:-1]
    speed_subset_start = np.sqrt(np.abs(u_subset_start)**2 + np.abs(v_subset_start)**2)
    speed_subset_end = np.sqrt(np.abs(u_subset_end)**2 + np.abs(v_subset_end)**2)

    # # create grids of coordinates rather than just lists
    lon_grid_u = lon_vec_u
    lat_grid_u = lat_vec_u
    lon_grid_u=lon_grid_u[:-1,:]
    lat_grid_u=lat_grid_u[:-1,:]

    # defining datetime objects for start and end time of simulation
    interval_type = 'seconds'
    interval_num_start = int(time_u[0])
    interval_num_end = int(time_u[-1])
    
    origin_time_u = datetime.datetime.strptime('2018-06-01 00:00:00','%Y-%m-%d %H:%M:%S') # simulation time output is number of hours since 00:00:00 01-01-1950
    start_time_u = origin_time_u + datetime.timedelta(**{interval_type: interval_num_start}) # turns integers into a datetime object to be added to origin_time
    end_time_u = origin_time_u + datetime.timedelta(**{interval_type: interval_num_end})
    
    # Normalize the data
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    
    m2 = ax.pcolormesh(lon_grid_u[:,:],lat_grid_u[:,:],speed_subset_end, cmap='Blues_r', norm=norm)#, vmin=-2, vmax =3) # mappable content
    ax.scatter(lon_p_end_SL[:], lat_p_end_SL[:],color='lightpink', s=0.01)
    ax.scatter(lon_p_end_India[:], lat_p_end_India[:],color='gold', s=0.01)
    ax.scatter(lon_p_end_Bang[:], lat_p_end_Bang[:],color='crimson', s=0.01)
    ax.scatter(lon_p_end_Myan[:], lat_p_end_Myan[:],color='mediumaquamarine', s=0.01)
    ax.scatter(lon_p_end_Thai[:], lat_p_end_Thai[:],color='k', s=0.01)
    ax.scatter(lon_p_end_Indo[:], lat_p_end_Indo[:],color='cyan', s=0.01)
#     m2.set_clim(0,max(np.amax(speed_subset_start),np.amax(speed_subset_end))) # same as setting vmin/vmax limits in countourf() or pcolormesh() line of code

    ax.set_yticks([])
    ax.set_yticklabels([])
    ax.xaxis.set_ticks_position('top')
    
    return m2

In [ ]:
def plot_particle_end_locs_ROMS_daily_noextras(ax, velocity_u_input_file, velocity_v_input_file, lon_p_end_SL, lon_p_end_India, lon_p_end_Bang, 
                                    lon_p_end_Myan, lon_p_end_Thai, lon_p_end_Indo, lat_p_end_SL, lat_p_end_India, 
                                               lat_p_end_Bang, lat_p_end_Myan, lat_p_end_Thai, lat_p_end_Indo, vmin, vmax):
    
    openfile_u = Dataset(velocity_u_input_file)
    openfile_v = Dataset(velocity_v_input_file)
    lon_vec_u = openfile_u.variables['lon_u']
    lat_vec_u = openfile_u.variables['lat_u']
    lon_vec_v = openfile_v.variables['lon_v']
    lat_vec_v = openfile_v.variables['lat_v']
    time_u = openfile_u.variables['ocean_time']
    u =  openfile_u.variables['u']
    v =  openfile_v.variables['v']
    
    u_subset_start = u[0,0,:-1,:]
    u_subset_end= u[-1,0,:-1,:]
    v_subset_start = v[0,0,:,:-1]
    v_subset_end = v[-1,0,:,:-1]
    speed_subset_start = np.sqrt(np.abs(u_subset_start)**2 + np.abs(v_subset_start)**2)
    speed_subset_end = np.sqrt(np.abs(u_subset_end)**2 + np.abs(v_subset_end)**2)

    # # create grids of coordinates rather than just lists
    lon_grid_u = lon_vec_u
    lat_grid_u = lat_vec_u
    lon_grid_u=lon_grid_u[:-1,:]
    lat_grid_u=lat_grid_u[:-1,:]

    # defining datetime objects for start and end time of simulation
    interval_type = 'days'
    interval_num_start = int(time_u[0])
    interval_num_end = int(time_u[-1])
    
    origin_time_u = datetime.datetime.strptime('2020-07-01 12:00:00','%Y-%m-%d %H:%M:%S') # simulation time output is number of hours since 00:00:00 01-01-1950
    start_time_u = origin_time_u + datetime.timedelta(**{interval_type: interval_num_start}) # turns integers into a datetime object to be added to origin_time
    end_time_u = origin_time_u + datetime.timedelta(**{interval_type: interval_num_end})
    
    # Normalize the data
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    
    m2 = ax.pcolormesh(lon_grid_u[:,:],lat_grid_u[:,:],speed_subset_end, cmap='Blues_r', norm=norm)#, vmin=-2, vmax =3) # mappable content
    ax.scatter(lon_p_end_SL[:], lat_p_end_SL[:],color='lightpink', s=0.01)
    ax.scatter(lon_p_end_India[:], lat_p_end_India[:],color='gold', s=0.01)
    ax.scatter(lon_p_end_Bang[:], lat_p_end_Bang[:],color='crimson', s=0.01)
    ax.scatter(lon_p_end_Myan[:], lat_p_end_Myan[:],color='mediumaquamarine', s=0.01)
    ax.scatter(lon_p_end_Thai[:], lat_p_end_Thai[:],color='k', s=0.01)
    ax.scatter(lon_p_end_Indo[:], lat_p_end_Indo[:],color='cyan', s=0.01)
    
    ax.set_xticks([])
    ax.set_xticklabels([])
    ax.set_yticks([])
    ax.set_yticklabels([])
    
    return m2

In [ ]:
def plot_conn_mat_diff_noextras(ax, input_conn_mat_file1, input_conn_mat_file2, vmin, vmax):
    conn_mat_df1 = pd.read_csv(input_conn_mat_file1, index_col=0)
    conn_mat_df2 = pd.read_csv(input_conn_mat_file2, index_col=0)
    conn_mat_df = conn_mat_df1 - conn_mat_df2
    conn_mat_df.replace(0.0, np.nan, inplace=True) # replace all zeros with NaNs, inplace=True changes it in the dataframe itself not in the variable created here.
    polygon_names_list_source = list(conn_mat_df1) # need to remove Andaman and Nicobar
    polygon_names_list_source = [polygon_names_list_source[x] for x in [0,1,2,3,4,5] ]
    polygon_names_list_sink = list(conn_mat_df1) # this prints out the column headings of the dataframe. For connectivity marices, the column and row headings are the same so I don't need to extract the row headings (which I think is more complicated...). I can just use the column headings as xticklabels and yticklabels

    #create figure
    m1 = ax.matshow(conn_mat_df, cmap='PRGn', vmin=vmin, vmax=vmax) # mappable content
    
    for (ii, jj), z in np.ndenumerate(conn_mat_df):
        if ~np.isnan(z):
            ax.text(jj, ii, '{:0.2f}'.format(z), ha='center', va='center')
 
    # Show all ticks and label them with the respective list entries
    ax.set_xticks(np.arange(len(polygon_names_list_sink)), labels=polygon_names_list_sink)
    ax.set_yticks(np.arange(len(polygon_names_list_source)), labels=polygon_names_list_source)
    ax.xaxis.set_ticks_position('bottom')
    # Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
         rotation_mode="anchor")

    if ax == axs[1,0]:
        ax.set_ylabel('Source', fontsize='x-large')
        ax.set_xticks([])
        ax.set_xticklabels([])
    elif ax == axs[2,1]:
        ax.set_xlabel('Sink', fontsize='x-large') 
        ax.set_yticks([])
        ax.set_yticklabels([])
    elif ax == axs[2,0]:
        ax.set_xlabel('Sink', fontsize='x-large')   
        ax.set_ylabel('Source', fontsize='x-large')
    elif ax == axs[1,1]:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])
    elif ax == axs[0,1]:
        ax.set_yticks([])
        ax.set_yticklabels([])
        ax.set_xticks([])
        ax.set_xticklabels([])
    elif ax == axs[0,0]:
        ax.set_xticks([])
        ax.set_xticklabels([])
        ax.set_ylabel('Source', fontsize='x-large')
        



In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(10, 12), dpi=300)
# Modify the axes that require a Cartopy projection
axs[0, 0] = plt.subplot(3, 2, 1, projection=ccrs.PlateCarree())
axs[0, 1] = plt.subplot(3, 2, 2, projection=ccrs.PlateCarree())
axs[1, 0] = plt.subplot(3, 2, 3, projection=ccrs.PlateCarree())
axs[1, 1] = plt.subplot(3, 2, 4, projection=ccrs.PlateCarree())
axs[0, 0].set_extent([77, 99, 3.5, 24])  # BoB domain
axs[0, 1].set_extent([77, 99, 3.5, 24])
axs[1, 0].set_extent([77, 99, 3.5, 24])
axs[1, 1].set_extent([77, 99, 3.5, 24])

m2 = plot_particle_end_locs_CMEMS_noextras(axs[0,0], velocity_input_file_Chourly, lon_p_end_SL_CH, lon_p_end_India_CH, lon_p_end_Bang_CH, 
                                    lon_p_end_Myan_CH, lon_p_end_Thai_CH, lon_p_end_Indo_CH, lat_p_end_SL_CH, lat_p_end_India_CH, lat_p_end_Bang_CH, 
                                          lat_p_end_Myan_CH, lat_p_end_Thai_CH, lat_p_end_Indo_CH, vminR, vmaxR)
plot_particle_end_locs_ROMS_hourly_noextras(axs[0,1], velocity_input_file_Rhourly, lon_p_end_SL_RH, lon_p_end_India_RH, lon_p_end_Bang_RH, 
                                    lon_p_end_Myan_RH, lon_p_end_Thai_RH, lon_p_end_Indo_RH, lat_p_end_SL_RH, lat_p_end_India_RH, lat_p_end_Bang_RH, 
                                     lat_p_end_Myan_RH, lat_p_end_Thai_RH, lat_p_end_Indo_RH, vminR, vmaxR)
plot_particle_end_locs_CMEMS_noextras1(axs[1,0], velocity_input_file_Cdaily, lon_p_end_SL_CD, lon_p_end_India_CD, lon_p_end_Bang_CD, 
                                    lon_p_end_Myan_CD, lon_p_end_Thai_CD, lon_p_end_Indo_CD, lat_p_end_SL_CD, lat_p_end_India_CD, lat_p_end_Bang_CD, 
                                      lat_p_end_Myan_CD, lat_p_end_Thai_CD, lat_p_end_Indo_CD, vminR, vmaxR)
plot_particle_end_locs_ROMS_daily_noextras(axs[1,1], velocity_u_input_file_Rdaily, velocity_v_input_file_Rdaily, lon_p_end_SL_RD, lon_p_end_India_RD, lon_p_end_Bang_RD, 
                                    lon_p_end_Myan_RD, lon_p_end_Thai_RD, lon_p_end_Indo_RD, lat_p_end_SL_RD, lat_p_end_India_RD, lat_p_end_Bang_RD, 
                                     lat_p_end_Myan_RD, lat_p_end_Thai_RD, lat_p_end_Indo_RD, vminR, vmaxR)
m1 = plot_conn_mat_diff_noextras(axs[2,0], Cop_hourly_July2020, Cop_daily_July2020, vmin=-0.063,vmax=0.063)
plot_conn_mat_diff_noextras(axs[2,1], ROMS_hourly_July2020, ROMS_daily_July2020, vmin=-0.063,vmax=0.063)

plt.subplots_adjust(left=0.1, right=0.9, bottom=0.2, top=0.9, wspace=0, hspace=None)

norm = plt.Normalize(vmin=-0.063, vmax=0.063)
cbar = fig.colorbar(m1, ax=axs[-1, :],location='bottom', cmap='PRGn', norm=norm)
# Get the current position of the colorbar
pos = cbar.ax.get_position()
# Set the new position of the colorbar
cbar.ax.set_position([pos.x0-0.06, pos.y0-0.29, pos.width+0.12, pos.height])

norm2 = plt.Normalize(vmin=vminR, vmax=vmaxR)
cbar2 = fig.colorbar(m2, ax=axs[-1, :],location='top', cmap='Blues', norm=norm2)
# Get the current position of the colorbar
pos2 = cbar.ax.get_position()
# Set the new position of the colorbar
cbar2.ax.set_position([pos.x0-0.06, pos.y0+0.745, pos.width+0.12, pos.height])
cbar2.set_label('m/s', fontsize=12, x=1.035, labelpad=-32)

axs[0,0].text(0.92, 0.98, "(a)", transform=axs[0,0].transAxes, fontsize=12, fontweight='bold', va='top')
axs[0,1].text(0.92, 0.98, "(b)", transform=axs[0,1].transAxes, fontsize=12, fontweight='bold', va='top')
axs[1,0].text(0.92, 0.98, "(c)", transform=axs[1,0].transAxes, fontsize=12, fontweight='bold', va='top')
axs[1,1].text(0.92, 0.98, "(d)", transform=axs[1,1].transAxes, fontsize=12, fontweight='bold', va='top')
axs[2,0].text(0.91, 0.98, "(e)", transform=axs[2,0].transAxes, fontsize=12, fontweight='bold', va='top')
axs[2,1].text(0.92, 0.98, "(f)", transform=axs[2,1].transAxes, fontsize=12, fontweight='bold', va='top')

axs[0,0].text(0.5, 1.4, "CMEMS", transform=axs[0,0].transAxes, fontsize=12, ha='center', va='top')
axs[0,1].text(0.5, 1.4, "ROMS", transform=axs[0,1].transAxes, fontsize=12, ha='center', va='top')

# Adjust spacing between subplots
plt.tight_layout()

# axs[0, 0].coastlines()  # Add coastlines to the subplot
gl1 = axs[0, 0].gridlines(draw_labels=True)  # Add gridlines and enable labels
gl1.bottom_labels = False  # Disable labels at the top
gl1.right_labels = False  # Disable labels on the right

# axs[0, 1].coastlines()  # Add coastlines to the subplot
gl2 = axs[0, 1].gridlines(draw_labels=True)  # Add gridlines and enable labels
gl2.bottom_labels = False  # Disable labels at the top
gl2.right_labels = False  # Disable labels on the right
gl2.left_labels = False

# axs[1, 0].coastlines()  # Add coastlines to the subplot
gl3 = axs[1, 0].gridlines(draw_labels=True)  # Add gridlines and enable labels
gl3.top_labels = False  # Disable labels at the top
gl3.right_labels = False  # Disable labels on the right
gl3.bottom_labels = False

# axs[1, 1].coastlines()  # Add coastlines to the subplot
gl4 = axs[1, 1].gridlines(draw_labels=True)  # Add gridlines and enable labels
gl4.top_labels = False  # Disable labels at the top
gl4.right_labels = False  # Disable labels on the right
gl4.left_labels = False
gl4.bottom_labels = False
fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/figB01", bbox_inches='tight', facecolor='white', transparent=False)


In [ ]:
# Figure C1

In [ ]:
fig = plt.figure(figsize=(20, 30))  # Adjust the overall figure size as needed

# Create a GridSpec with 5 rows and 2 columns
gs = gridspec.GridSpec(5, 2, height_ratios=[1.2, 3.2, 2.2, 2.1, 1.5])  # Adjust height ratios as needed

# Create subplots and assign them to the GridSpec
ax1 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
ax2 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
ax3 = fig.add_subplot(gs[1, 0], projection=ccrs.PlateCarree())
ax4 = fig.add_subplot(gs[1, 1], projection=ccrs.PlateCarree())
ax5 = fig.add_subplot(gs[2, 0], projection=ccrs.PlateCarree())
ax6 = fig.add_subplot(gs[2, 1], projection=ccrs.PlateCarree())
ax7 = fig.add_subplot(gs[3, 0], projection=ccrs.PlateCarree())
ax8 = fig.add_subplot(gs[3, 1], projection=ccrs.PlateCarree())
ax9 = fig.add_subplot(gs[4, 0], projection=ccrs.PlateCarree())
ax10 = fig.add_subplot(gs[4, 1], projection=ccrs.PlateCarree())

ax1.set_extent([92,99,7,11]) # pink
gl1 = ax1.gridlines(draw_labels=True)
gl1.xlabel_style = {'fontsize': 10}
gl1.ylabel_style = {'fontsize': 10}
gl1.right_labels = False
gl1.top_labels = False

m1 = ax1.pcolormesh(lon_grid_u,lat_grid_u,speed_subset_cyan, cmap='Blues_r', vmin=np.amin(speed_subset_cyan), vmax=np.amax(speed_subset_cyan)) # mappable content

for ii in range(0,lon_p1.shape[0],4):
    if 0 <= time_p1[ii,0] < 604800:
        ax1.scatter(lon_p1[ii,:8], lat_p1[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(1,lon_p1.shape[0],4):
    if 604800 <= time_p1[ii,0] < 1209600:
        ax1.scatter(lon_p1[ii,:8], lat_p1[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(2,lon_p1.shape[0],4):
    if 1209600 <= time_p1[ii,0] < 1814400:
        ax1.scatter(lon_p1[ii,:8], lat_p1[ii,:8], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=1)

p1 = ax1.scatter(lon_drifter4[1:], lat_drifter4[1:], transform=ccrs.PlateCarree(), color='lightpink', s=2)
ax1.scatter(lon_drifter4[0:1], lat_drifter4[0:1], transform=ccrs.PlateCarree(), color='lightpink', s=10, marker='*')

ax2.set_extent([92,99,7,11]) # pink
gl2 = ax2.gridlines(draw_labels=True)
gl2.xlabel_style = {'fontsize': 10}
gl2.ylabel_style = {'fontsize': 10}
gl2.right_labels = False
gl2.top_labels = False

m2 = ax2.pcolormesh(lon_grid_uR,lat_grid_uR,speed_subsetR_cyan, cmap='Blues_r', vmin=np.amin(speed_subsetR_cyan), vmax=np.amax(speed_subsetR_cyan)) # mappable content

for ii in range(0,lon_pR1.shape[0],4):
    if 0 <= time_pR1[ii,0] < 604800:
        ax2.scatter(lon_pR1[ii,:8], lat_pR1[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(1,lon_pR1.shape[0],4):
    if 604800 <= time_pR1[ii,0] < 1209600:
        ax2.scatter(lon_pR1[ii,:8], lat_pR1[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(2,lon_pR1.shape[0],4):
    if 1209600 <= time_pR1[ii,0] < 1814400:
        ax2.scatter(lon_pR1[ii,:8], lat_pR1[ii,:8], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=1)

p2 = ax2.scatter(lon_drifter4[1:], lat_drifter4[1:], transform=ccrs.PlateCarree(), color='lightpink', s=2)
ax2.scatter(lon_drifter4[0:1], lat_drifter4[0:1], transform=ccrs.PlateCarree(), color='lightpink', s=10, marker='*')

ax3.set_extent([90,94,14,20]) # gold
gl3 = ax3.gridlines(draw_labels=True)
gl3.xlabel_style = {'fontsize': 10}
gl3.ylabel_style = {'fontsize': 10}
gl3.right_labels = False
gl3.top_labels = False

m3 = ax3.pcolormesh(lon_grid_u,lat_grid_u,speed_subset_cyan, cmap='Blues_r', vmin=np.amin(speed_subset_cyan), vmax=np.amax(speed_subset_cyan)) # mappable content

for ii in range(0,lon_p2.shape[0],11):
    if 0 <= time_p2[ii,0] < 604800:
        ax3.scatter(lon_p2[ii,:8], lat_p2[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(1,lon_p2.shape[0],11):
    if 604800 <= time_p2[ii,0] < 1209600:
        ax3.scatter(lon_p2[ii,:8], lat_p2[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(2,lon_p2.shape[0],11):
    if 1209600 <= time_p2[ii,0] < 1814400:
        ax3.scatter(lon_p2[ii,:8], lat_p2[ii,:8], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=1)
for ii in range(3,lon_p2.shape[0],11):
    if 1814400 <= time_p2[ii,0] < 2419200:
        ax3.scatter(lon_p2[ii,:8], lat_p2[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(4,lon_p2.shape[0],11):
    if 2419200 <= time_p2[ii,0] < 3024000:
        ax3.scatter(lon_p2[ii,:8], lat_p2[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(5,lon_p2.shape[0],11):
    if 3024000 <= time_p2[ii,0] < 3628800:
        ax3.scatter(lon_p2[ii,:8], lat_p2[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(6,lon_p2.shape[0],11):
    if 3628800 <= time_p2[ii,0] < 4233600:
        ax3.scatter(lon_p2[ii,:8], lat_p2[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(7,lon_p2.shape[0],11):
    if 4233600 <= time_p2[ii,0] < 4838400:
        ax3.scatter(lon_p2[ii,:8], lat_p2[ii,:8], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=1)
for ii in range(8,lon_p2.shape[0],11):
    if 4838400 <= time_p2[ii,0] < 5443200:
        ax3.scatter(lon_p2[ii,:8], lat_p2[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(9,lon_p2.shape[0],11):
    if 5443200 <= time_p2[ii,0] < 6048000:
        ax3.scatter(lon_p2[ii,:8], lat_p2[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)

p3 = ax3.scatter(lon_drifter5[1:], lat_drifter5[1:], transform=ccrs.PlateCarree(), color='gold', s=2)
ax3.scatter(lon_drifter5[0:1], lat_drifter5[0:1], transform=ccrs.PlateCarree(), color='gold', s=10, marker='*')

ax4.set_extent([90,94,14,20]) # gold
gl4 = ax4.gridlines(draw_labels=True)
gl4.xlabel_style = {'fontsize': 10}
gl4.ylabel_style = {'fontsize': 10}
gl4.right_labels = False
gl4.top_labels = False

m4 = ax4.pcolormesh(lon_grid_uR,lat_grid_uR,speed_subsetR_cyan, cmap='Blues_r', vmin=np.amin(speed_subsetR_cyan), vmax=np.amax(speed_subsetR_cyan)) # mappable content

for ii in range(0,lon_pR2.shape[0],11):
    if 0 <= time_p2[ii,0] < 604800:
        ax4.scatter(lon_pR2[ii,:8], lat_pR2[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(1,lon_pR2.shape[0],11):
    if 604800 <= time_pR2[ii,0] < 1209600:
        ax4.scatter(lon_pR2[ii,:8], lat_pR2[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(2,lon_pR2.shape[0],11):
    if 1209600 <= time_pR2[ii,0] < 1814400:
        ax4.scatter(lon_pR2[ii,:8], lat_pR2[ii,:8], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=1)
for ii in range(3,lon_pR2.shape[0],11):
    if 1814400 <= time_pR2[ii,0] < 2419200:
        ax4.scatter(lon_pR2[ii,:8], lat_pR2[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(4,lon_pR2.shape[0],11):
    if 2419200 <= time_pR2[ii,0] < 3024000:
        ax4.scatter(lon_pR2[ii,:8], lat_pR2[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(5,lon_pR2.shape[0],11):
    if 3024000 <= time_pR2[ii,0] < 3628800:
        ax4.scatter(lon_pR2[ii,:8], lat_pR2[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(6,lon_pR2.shape[0],11):
    if 3628800 <= time_pR2[ii,0] < 4233600:
        ax4.scatter(lon_pR2[ii,:8], lat_pR2[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(7,lon_pR2.shape[0],11):
    if 4233600 <= time_pR2[ii,0] < 4838400:
        ax4.scatter(lon_pR2[ii,:8], lat_pR2[ii,:8], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=1)
for ii in range(8,lon_pR2.shape[0],11):
    if 4838400 <= time_pR2[ii,0] < 5443200:
        ax4.scatter(lon_pR2[ii,:8], lat_pR2[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(9,lon_pR2.shape[0],11):
    if 5443200 <= time_pR2[ii,0] < 6048000:
        ax4.scatter(lon_pR2[ii,:8], lat_pR2[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)

p4 = ax4.scatter(lon_drifter5[1:], lat_drifter5[1:], transform=ccrs.PlateCarree(), color='gold', s=2)
ax4.scatter(lon_drifter5[0:1], lat_drifter5[0:1], transform=ccrs.PlateCarree(), color='gold', s=10, marker='*')

ax5.set_extent([91,97,6,12]) # orange
gl5 = ax5.gridlines(draw_labels=True)
gl5.xlabel_style = {'fontsize': 10}
gl5.ylabel_style = {'fontsize': 10}
gl5.right_labels = False
gl5.top_labels = False

m5 = ax5.pcolormesh(lon_grid_u,lat_grid_u,speed_subset_cyan, cmap='Blues_r', vmin=np.amin(speed_subset_cyan), vmax=np.amax(speed_subset_cyan)) # mappable content

for ii in range(0,lon_p3.shape[0],9):
    if 0 <= time_p3[ii,0] < 604800:
        ax5.scatter(lon_p3[ii,:8], lat_p3[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(1,lon_p3.shape[0],9):
    if 604800 <= time_p3[ii,0] < 1209600:
        ax5.scatter(lon_p3[ii,:8], lat_p3[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(2,lon_p3.shape[0],9):
    if 1209600 <= time_p3[ii,0] < 1814400:
        ax5.scatter(lon_p3[ii,:8], lat_p3[ii,:8], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=1)
for ii in range(3,lon_p3.shape[0],9):
    if 1814400 <= time_p3[ii,0] < 2419200:
        ax5.scatter(lon_p3[ii,:8], lat_p3[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(4,lon_p3.shape[0],9):
    if 2419200 <= time_p3[ii,0] < 3024000:
        ax5.scatter(lon_p3[ii,:8], lat_p3[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(5,lon_p3.shape[0],9):
    if 3024000 <= time_p3[ii,0] < 3628800:
        ax5.scatter(lon_p3[ii,:8], lat_p3[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(6,lon_p3.shape[0],9):
    if 3628800 <= time_p3[ii,0] < 4233600:
        ax5.scatter(lon_p3[ii,:8], lat_p3[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(7,lon_p3.shape[0],9):
    if 4233600 <= time_p3[ii,0] < 4838400:
        ax5.scatter(lon_p3[ii,:8], lat_p3[ii,:8], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=1)

p5 = ax5.scatter(lon_drifter6[1:], lat_drifter6[1:], transform=ccrs.PlateCarree(), color='crimson', s=2)
ax5.scatter(lon_drifter6[0:1], lat_drifter6[0:1], transform=ccrs.PlateCarree(), color='crimson', s=10, marker='*')

ax6.set_extent([91,97,6,12]) # crimson
gl6 = ax6.gridlines(draw_labels=True)
gl6.xlabel_style = {'fontsize': 10}
gl6.ylabel_style = {'fontsize': 10}
gl6.right_labels = False
gl6.top_labels = False

m6 = ax6.pcolormesh(lon_grid_uR,lat_grid_uR,speed_subsetR_cyan, cmap='Blues_r', vmin=np.amin(speed_subsetR_cyan), vmax=np.amax(speed_subsetR_cyan)) # mappable content

for ii in range(0,lon_pR3.shape[0],9):
    if 0 <= time_pR3[ii,0] < 604800:
        ax6.scatter(lon_pR3[ii,:8], lat_pR3[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(1,lon_pR3.shape[0],9):
    if 604800 <= time_pR3[ii,0] < 1209600:
        ax6.scatter(lon_pR3[ii,:8], lat_pR3[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(2,lon_pR3.shape[0],9):
    if 1209600 <= time_pR3[ii,0] < 1814400:
        ax6.scatter(lon_pR3[ii,:8], lat_pR3[ii,:8], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=1)
for ii in range(3,lon_pR3.shape[0],9):
    if 1814400 <= time_pR3[ii,0] < 2419200:
        ax6.scatter(lon_pR3[ii,:8], lat_pR3[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(4,lon_pR3.shape[0],9):
    if 2419200 <= time_pR3[ii,0] < 3024000:
        ax6.scatter(lon_pR3[ii,:8], lat_pR3[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(5,lon_pR3.shape[0],9):
    if 3024000 <= time_pR3[ii,0] < 3628800:
        ax6.scatter(lon_pR3[ii,:8], lat_pR3[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(6,lon_pR3.shape[0],9):
    if 3628800 <= time_pR3[ii,0] < 4233600:
        ax6.scatter(lon_pR3[ii,:8], lat_pR3[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(7,lon_pR3.shape[0],9):
    if 4233600 <= time_pR3[ii,0] < 4838400:
        ax6.scatter(lon_pR3[ii,:8], lat_pR3[ii,:8], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=1)

p6 = ax6.scatter(lon_drifter6[2:], lat_drifter6[2:], transform=ccrs.PlateCarree(), color='crimson', s=2)
ax6.scatter(lon_drifter6[0:1], lat_drifter6[0:1], transform=ccrs.PlateCarree(), color='crimson', s=10, marker='*')

ax7.set_extent([84,99,8,22]) # marine
gl7 = ax7.gridlines(draw_labels=True)
gl7.xlabel_style = {'fontsize': 10}
gl7.ylabel_style = {'fontsize': 10}
gl7.right_labels = False
gl7.top_labels = False

m7 = ax7.pcolormesh(lon_grid_u,lat_grid_u,speed_subset_cyan, cmap='Blues_r', vmin=np.amin(speed_subset_cyan), vmax=np.amax(speed_subset_cyan)) # mappable content

for ii in range(0,lon_p4.shape[0],46):
    if 0 <= time_p4[ii,0] < 604800:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(1,lon_p4.shape[0],46):
    if 604800 <= time_p4[ii,0] < 1209600:
        ax7.scatter(lon_p4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(2,lon_p4.shape[0],46):
    if 1209600 <= time_p4[ii,0] < 1814400:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(3,lon_p4.shape[0],46):
    if 1814400 <= time_p4[ii,0] < 2419200:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(4,lon_p4.shape[0],46):
    if 2419200 <= time_p4[ii,0] < 3024000:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(5,lon_p4.shape[0],46):
    if 3024000 <= time_p4[ii,0] < 3628800:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(6,lon_p4.shape[0],46):
    if 3628800 <= time_p4[ii,0] < 4233600:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(7,lon_p4.shape[0],46):
    if 4233600 <= time_p4[ii,0] < 4838400:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(8,lon_p4.shape[0],46):
    if 4838400 <= time_p4[ii,0] < 5443200:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(9,lon_p4.shape[0],46):
    if 5443200 <= time_p4[ii,0] < 6048000:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(10,lon_p4.shape[0],46):
    if 6048000 <= time_p4[ii,0] < 6652800:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(11,lon_p4.shape[0],46):
    if 6652800 <= time_p4[ii,0] < 7257600:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(12,lon_p4.shape[0],46):
    if 7257600 <= time_p4[ii,0] < 7862400:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(13,lon_p4.shape[0],46):
    if 7862400 <= time_p4[ii,0] < 8467200:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(14,lon_p4.shape[0],46):
    if 8467200 <= time_p4[ii,0] < 9072000:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(15,lon_p4.shape[0],46):
    if 9072000 <= time_p4[ii,0] < 9676800:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(16,lon_p4.shape[0],46):
    if 10281600 <= time_p4[ii,0] < 10281600:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(17,lon_p4.shape[0],46):
    if 10886400 <= time_p4[ii,0] < 10886400:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(18,lon_p4.shape[0],46):
    if 11491200 <= time_p4[ii,0] < 11491200:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(19,lon_p4.shape[0],46):
    if 12096000 <= time_p4[ii,0] < 12096000:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(20,lon_p4.shape[0],46):
    if 12700800 <= time_p4[ii,0] < 12700800:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(21,lon_p4.shape[0],46):
    if 13305600 <= time_p4[ii,0] < 13305600:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(22,lon_p4.shape[0],46):
    if 13305600 <= time_p4[ii,0] < 13910400:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(23,lon_p4.shape[0],46):
    if 13910400 <= time_p4[ii,0] < 14515200:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(24,lon_p4.shape[0],46):
    if 14515200 <= time_p4[ii,0] < 15120000:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(25,lon_p4.shape[0],46):
    if 15120000 <= time_p4[ii,0] < 15724800:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(26,lon_p4.shape[0],46):
    if 15724800 <= time_p4[ii,0] < 16329600:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(27,lon_p4.shape[0],46):
    if 16329600 <= time_p4[ii,0] < 16934400:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(28,lon_p4.shape[0],46):
    if 16934400 <= time_p4[ii,0] < 17539200:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(29,lon_p4.shape[0],46):
    if 17539200 <= time_p4[ii,0] < 18144000:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(30,lon_p4.shape[0],46):
    if 18144000 <= time_p4[ii,0] < 18748800:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(31,lon_p4.shape[0],46):
    if 18748800 <= time_p4[ii,0] < 19353600:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(32,lon_p4.shape[0],46):
    if 19353600 <= time_p4[ii,0] < 19958400:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(33,lon_p4.shape[0],46):
    if 19958400 <= time_p4[ii,0] < 20563200:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(34,lon_p4.shape[0],46):
    if 20563200 <= time_p4[ii,0] < 21168000:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(35,lon_p4.shape[0],46):
    if 21168000 <= time_p4[ii,0] < 21772800:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(36,lon_p4.shape[0],46):
    if 21772800 <= time_p4[ii,0] < 22377600:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(37,lon_p4.shape[0],46):
    if 22377600 <= time_p4[ii,0] < 22982400:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(38,lon_p4.shape[0],46):
    if 22982400 <= time_p4[ii,0] < 23587200:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(39,lon_p4.shape[0],46):
    if 23587200 <= time_p4[ii,0] < 24192000:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(40,lon_p4.shape[0],46):
    if 24192000 <= time_p4[ii,0] < 24796800:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(41,lon_p4.shape[0],46):
    if 24796800 <= time_p4[ii,0] < 25401600:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(42,lon_p4.shape[0],46):
    if 25401600 <= time_p4[ii,0] < 26006400:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(43,lon_p4.shape[0],46):
    if 26006400 <= time_p4[ii,0] < 26611200:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(44,lon_p4.shape[0],46):
    if 26611200 <= time_p4[ii,0] < 27216000:
        ax7.scatter(lon_p4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)

p7 = ax7.scatter(lon_drifter7[2:], lat_drifter7[2:], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=2)
ax7.scatter(lon_drifter7[0:1], lat_drifter7[0:1], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=10, marker='*')

ax8.set_extent([84,99,8,22]) # marine
gl8 = ax8.gridlines(draw_labels=True)
gl8.xlabel_style = {'fontsize': 10}
gl8.ylabel_style = {'fontsize': 10}
gl8.right_labels = False
gl8.top_labels = False

m8 = ax8.pcolormesh(lon_grid_uR,lat_grid_uR,speed_subsetR_cyan, cmap='Blues_r', vmin=np.amin(speed_subsetR_cyan), vmax=np.amax(speed_subsetR_cyan)) # mappable content

for ii in range(0,lon_pR4.shape[0],46):
    if 0 <= time_pR4[ii,0] < 604800:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(1,lon_pR4.shape[0],46):
    if 604800 <= time_pR4[ii,0] < 1209600:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(2,lon_pR4.shape[0],46):
    if 1209600 <= time_pR4[ii,0] < 1814400:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(3,lon_pR4.shape[0],46):
    if 1814400 <= time_pR4[ii,0] < 2419200:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(4,lon_pR4.shape[0],46):
    if 2419200 <= time_pR4[ii,0] < 3024000:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(5,lon_pR4.shape[0],46):
    if 3024000 <= time_pR4[ii,0] < 3628800:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(6,lon_pR4.shape[0],46):
    if 3628800 <= time_pR4[ii,0] < 4233600:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(7,lon_pR4.shape[0],46):
    if 4233600 <= time_pR4[ii,0] < 4838400:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(8,lon_pR4.shape[0],46):
    if 4838400 <= time_pR4[ii,0] < 5443200:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(9,lon_pR4.shape[0],46):
    if 5443200 <= time_pR4[ii,0] < 6048000:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(10,lon_pR4.shape[0],46):
    if 6048000 <= time_pR4[ii,0] < 6652800:
        ax8.scatter(lon_pR4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(11,lon_pR4.shape[0],46):
    if 6652800 <= time_pR4[ii,0] < 7257600:
        ax8.scatter(lon_pR4[ii,:8], lat_p4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(12,lon_pR4.shape[0],46):
    if 7257600 <= time_pR4[ii,0] < 7862400:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(13,lon_pR4.shape[0],46):
    if 7862400 <= time_pR4[ii,0] < 8467200:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(14,lon_pR4.shape[0],46):
    if 8467200 <= time_pR4[ii,0] < 9072000:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(15,lon_pR4.shape[0],46):
    if 9072000 <= time_pR4[ii,0] < 9676800:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(16,lon_pR4.shape[0],46):
    if 10281600 <= time_pR4[ii,0] < 10281600:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(17,lon_pR4.shape[0],46):
    if 10886400 <= time_pR4[ii,0] < 10886400:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(18,lon_pR4.shape[0],46):
    if 11491200 <= time_pR4[ii,0] < 11491200:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(19,lon_pR4.shape[0],46):
    if 12096000 <= time_pR4[ii,0] < 12096000:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(20,lon_pR4.shape[0],46):
    if 12700800 <= time_pR4[ii,0] < 12700800:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(21,lon_pR4.shape[0],46):
    if 13305600 <= time_pR4[ii,0] < 13305600:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(22,lon_pR4.shape[0],46):
    if 13305600 <= time_pR4[ii,0] < 13910400:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(23,lon_pR4.shape[0],46):
    if 13910400 <= time_pR4[ii,0] < 14515200:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(24,lon_pR4.shape[0],46):
    if 14515200 <= time_pR4[ii,0] < 15120000:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(25,lon_pR4.shape[0],46):
    if 15120000 <= time_pR4[ii,0] < 15724800:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(26,lon_pR4.shape[0],46):
    if 15724800 <= time_pR4[ii,0] < 16329600:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(27,lon_pR4.shape[0],46):
    if 16329600 <= time_pR4[ii,0] < 16934400:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(28,lon_pR4.shape[0],46):
    if 16934400 <= time_pR4[ii,0] < 17539200:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(29,lon_pR4.shape[0],46):
    if 17539200 <= time_pR4[ii,0] < 18144000:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(30,lon_pR4.shape[0],46):
    if 18144000 <= time_pR4[ii,0] < 18748800:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(31,lon_pR4.shape[0],46):
    if 18748800 <= time_pR4[ii,0] < 19353600:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(32,lon_pR4.shape[0],46):
    if 19353600 <= time_pR4[ii,0] < 19958400:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(33,lon_pR4.shape[0],46):
    if 19958400 <= time_pR4[ii,0] < 20563200:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(34,lon_pR4.shape[0],46):
    if 20563200 <= time_pR4[ii,0] < 21168000:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(35,lon_pR4.shape[0],46):
    if 21168000 <= time_pR4[ii,0] < 21772800:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(36,lon_pR4.shape[0],46):
    if 21772800 <= time_pR4[ii,0] < 22377600:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(37,lon_pR4.shape[0],46):
    if 22377600 <= time_pR4[ii,0] < 22982400:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(38,lon_pR4.shape[0],46):
    if 22982400 <= time_pR4[ii,0] < 23587200:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(39,lon_pR4.shape[0],46):
    if 23587200 <= time_pR4[ii,0] < 24192000:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)
for ii in range(40,lon_pR4.shape[0],46):
    if 24192000 <= time_pR4[ii,0] < 24796800:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(41,lon_pR4.shape[0],46):
    if 24796800 <= time_pR4[ii,0] < 25401600:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(42,lon_pR4.shape[0],46):
    if 25401600 <= time_pR4[ii,0] < 26006400:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(43,lon_pR4.shape[0],46):
    if 26006400 <= time_pR4[ii,0] < 26611200:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)
for ii in range(44,lon_pR4.shape[0],46):
    if 26611200 <= time_pR4[ii,0] < 27216000:
        ax8.scatter(lon_pR4[ii,:8], lat_pR4[ii,:8], transform=ccrs.PlateCarree(), color='cyan', s=1)

p8 = ax8.scatter(lon_drifter7[2:], lat_drifter7[2:], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=2)
ax8.scatter(lon_drifter7[0:1], lat_drifter7[0:1], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=10, marker='*')

ax9.set_extent([85,94,9,15]) # cyan 
gl9 = ax9.gridlines(draw_labels=True)
gl9.xlabel_style = {'fontsize': 10}
gl9.ylabel_style = {'fontsize': 10}
gl9.right_labels = False
gl9.top_labels = False

m9 = ax9.pcolormesh(lon_grid_u,lat_grid_u,speed_subset_cyan, cmap='Blues_r', vmin=np.amin(speed_subset_cyan), vmax=np.amax(speed_subset_cyan)) # mappable content

for ii in range(0,lon_p5.shape[0],6): # from first particle to final particle, in steps of 6 because there were 5 complete weeks
    if 0 <= time_p5[ii,0] < 604800:
        ax9.scatter(lon_p5[ii,:8], lat_p5[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(1,lon_p5.shape[0],6):
    if 604800 <= time_p5[ii,0] < 1209600:
        ax9.scatter(lon_p5[ii,:8], lat_p5[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(2,lon_p5.shape[0],6):
    if 1209600 <= time_p5[ii,0] < 1814400:
        ax9.scatter(lon_p5[ii,:8], lat_p5[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(3,lon_p5.shape[0],6):
    if 1814400 <= time_p5[ii,0] < 2419200:
        ax9.scatter(lon_p5[ii,:8], lat_p5[ii,:8], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=1)
for ii in range(4,lon_p5.shape[0],6):
    if 2419200 <= time_p5[ii,0] < 3024000:
        ax9.scatter(lon_p5[ii,:8], lat_p5[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)

p9 = ax9.scatter(lon_drifter8[2:], lat_drifter8[2:], transform=ccrs.PlateCarree(), color='cyan', s=2)
ax9.scatter(lon_drifter8[0:1], lat_drifter8[0:1], transform=ccrs.PlateCarree(), color='cyan', s=10, marker='*')

ax10.set_extent([85,94,9,15]) # cyan
gl10 = ax10.gridlines(draw_labels=True)
gl10.xlabel_style = {'fontsize': 10}
gl10.ylabel_style = {'fontsize': 10}
gl10.right_labels = False
gl10.top_labels = False

m10 = ax10.pcolormesh(lon_grid_uR,lat_grid_uR,speed_subsetR_cyan, cmap='Blues_r', vmin=np.amin(speed_subsetR_cyan), vmax=np.amax(speed_subsetR_cyan)) # mappable content
    
for ii in range(0,lon_pR5.shape[0],6): # from first particle to final particle, in steps of 6 because there were 5 complete weeks
    if 0 <= time_pR5[ii,0] < 604800:
        ax10.scatter(lon_pR5[ii,:8], lat_pR5[ii,:8], transform=ccrs.PlateCarree(), color='lightpink', s=1)
for ii in range(1,lon_pR5.shape[0],6):
    if 604800 <= time_pR5[ii,0] < 1209600:
        ax10.scatter(lon_pR5[ii,:8], lat_pR5[ii,:8], transform=ccrs.PlateCarree(), color='gold', s=1)
for ii in range(2,lon_pR5.shape[0],6):
    if 1209600 <= time_pR5[ii,0] < 1814400:
        ax10.scatter(lon_pR5[ii,:8], lat_pR5[ii,:8], transform=ccrs.PlateCarree(), color='crimson', s=1)
for ii in range(3,lon_pR5.shape[0],6):
    if 1814400 <= time_pR5[ii,0] < 2419200:
        ax10.scatter(lon_pR5[ii,:8], lat_pR5[ii,:8], transform=ccrs.PlateCarree(), color='mediumaquamarine', s=1)
for ii in range(4,lon_pR5.shape[0],6):
    if 2419200 <= time_pR5[ii,0] < 3024000:
        ax10.scatter(lon_pR5[ii,:8], lat_pR5[ii,:8], transform=ccrs.PlateCarree(), color='k', s=1)

p10 = ax10.scatter(lon_drifter8[2:], lat_drifter8[2:], transform=ccrs.PlateCarree(), color='cyan', s=2)
ax10.scatter(lon_drifter8[0:1], lat_drifter8[0:1], transform=ccrs.PlateCarree(), color='cyan', s=10, marker='*')

ax1.text(0, 1.155, "(a)", transform=ax1.transAxes, fontsize=12, fontweight='bold', va='top')
ax2.text(0.87, 1.155, "(b)", transform=ax2.transAxes, fontsize=12, fontweight='bold', va='top')
ax3.text(0, 1.06, "(c)", transform=ax3.transAxes, fontsize=12, fontweight='bold', va='top')
ax4.text(0.87, 1.06, "(d)", transform=ax4.transAxes, fontsize=12, fontweight='bold', va='top')
ax5.text(0, 1.090, "(e)", transform=ax5.transAxes, fontsize=12, fontweight='bold', va='top')
ax6.text(0.87, 1.090, "(f)", transform=ax6.transAxes, fontsize=12, fontweight='bold', va='top')
ax7.text(0, 1.095, "(g)", transform=ax7.transAxes, fontsize=12, fontweight='bold', va='top')
ax8.text(0.87, 1.095, "(h)", transform=ax8.transAxes, fontsize=12, fontweight='bold', va='top')
ax9.text(0, 1.125, "(i)", transform=ax9.transAxes, fontsize=12, fontweight='bold', va='top')
ax10.text(0.87, 1.125, "(j)", transform=ax10.transAxes, fontsize=12, fontweight='bold', va='top')

ax1.text(0.48, 1.20, "CMEMS", transform=ax1.transAxes, fontsize=15, ha='center', va='top')
ax2.text(0.48, 1.20, "ROMS", transform=ax2.transAxes, fontsize=15, ha='center', va='top')

# Adjust spacing between subplots
plt.subplots_adjust(left=0.35, right=0.65, bottom=0.3, top=0.7, wspace=0.3, hspace=0.3)

ax1.text(1.22, 1.06, 'D1', ha='center', va='center', fontsize=14, transform=ax1.transAxes)
ax3.text(1.22, 1.03, 'D2', ha='center', va='center', fontsize=14, transform=ax3.transAxes)
ax5.text(1.22, 1.04, 'D3', ha='center', va='center', fontsize=14, transform=ax5.transAxes)
ax7.text(1.22, 1.05, 'D4', ha='center', va='center', fontsize=14, transform=ax7.transAxes)
ax9.text(1.22, 1.05, 'D5', ha='center', va='center', fontsize=14, transform=ax9.transAxes)

# Add the first colorbar on the left
cbar1 = fig.colorbar(m1, ax=[ax1,ax3,ax5,ax7,ax9], location='left', shrink=0.6)
cbar1.set_label('m/s',fontsize=12)

# Add the second colorbar on the right
cbar2 = fig.colorbar(m2, ax=[ax2,ax4,ax6,ax8,ax10], location='right', shrink=0.6)
cbar2.set_label('m/s',fontsize=12)

# Adjust the position of the colorbars
cbar1.ax.set_position([0.33, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
cbar2.ax.set_position([0.64, 0.15, 0.02, 0.7])  # [left, bottom, width, height]

fig.savefig("/gpfs/home/rpe16nbu/projects/ocpp_mo1/plots/paper_data/figC1", bbox_inches='tight', facecolor='white', transparent=False)
